In [1]:
# =============================================================================
# CELL 1 — DIM_CUSTOMER SOURCE CONFIGURATION AND VALIDATION
#
# Purpose:
# - Define Bronze and Silver objects used by dim_customer.
# - Confirm that all live and legacy customer sources are available.
# - Confirm that dbo.dim_company exists in Silver.
# - Create one fixed run ID and timestamp for the complete notebook execution.
#
# Notebook requirements:
# - lh_global_finance_silver must be the default Lakehouse.
# - lh_global_finance_bronze must be attached.
# =============================================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)


# -----------------------------------------------------------------------------
# 1. Environment configuration
# -----------------------------------------------------------------------------

WORKSPACE_NAME = "Global-Finance-Analytics-DEV"

BRONZE_LAKEHOUSE = "lh_global_finance_bronze"
BRONZE_SCHEMA = "dbo"

SILVER_LAKEHOUSE = "lh_global_finance_silver"
SILVER_SCHEMA = "dbo"


DIM_CUSTOMER_RUN_STARTED_UTC = datetime.now(
    timezone.utc
).replace(tzinfo=None)


PIPELINE_RUN_ID = (
    "dim_customer_manual_"
    + datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%S"
    )
)


# -----------------------------------------------------------------------------
# 2. Fully qualified table-name helpers
# -----------------------------------------------------------------------------

def qualified_table(
    lakehouse_name: str,
    schema_name: str,
    table_name: str,
) -> str:
    """
    Return a fully qualified Fabric table identifier.

    Backticks are required because the workspace name contains hyphens.
    """

    return (
        f"`{WORKSPACE_NAME}`."
        f"`{lakehouse_name}`."
        f"`{schema_name}`."
        f"`{table_name}`"
    )


def bronze_table(
    table_name: str,
) -> str:
    """
    Return a fully qualified Bronze table identifier.
    """

    return qualified_table(
        BRONZE_LAKEHOUSE,
        BRONZE_SCHEMA,
        table_name,
    )


def silver_table(
    table_name: str,
) -> str:
    """
    Return a fully qualified Silver table identifier.
    """

    return qualified_table(
        SILVER_LAKEHOUSE,
        SILVER_SCHEMA,
        table_name,
    )


# -----------------------------------------------------------------------------
# 3. Customer source configuration
#
# Priority rule:
# - LIVE_SOURCE / LIVE_API = priority 1
# - LEGACY_FILE = priority 2
#
# Lower number means higher priority.
# -----------------------------------------------------------------------------

CUSTOMER_SOURCE_TABLES = {
    "SAP_UK_LIVE": {
        "source_system": "SAP_UK",
        "source_company": "UK01",
        "record_origin": "LIVE_SOURCE",
        "source_priority": 1,
        "table_name": bronze_table(
            "sap_uk_ocrd"
        ),
    },

    "QBO_ES_LIVE": {
        "source_system": "QBO_ES",
        "source_company": "ES01",
        "record_origin": "LIVE_API",
        "source_priority": 1,
        "table_name": bronze_table(
            "qbo_es_customers"
        ),
    },

    "CZECH_ERP_LIVE": {
        "source_system": "CZECH_ERP",
        "source_company": "CZ01",
        "record_origin": "LIVE_SOURCE",
        "source_priority": 1,
        "table_name": bronze_table(
            "czech_erp_customers"
        ),
    },

    "SAP_UK_LEGACY": {
        "source_system": "SAP_UK",
        "source_company": "UK01",
        "record_origin": "LEGACY_FILE",
        "source_priority": 2,
        "table_name": bronze_table(
            "legacy_sap_uk_customers"
        ),
    },

    "QBO_ES_LEGACY": {
        "source_system": "QBO_ES",
        "source_company": "ES01",
        "record_origin": "LEGACY_FILE",
        "source_priority": 2,
        "table_name": bronze_table(
            "legacy_qbo_es_customers"
        ),
    },

    "CZECH_ERP_LEGACY": {
        "source_system": "CZECH_ERP",
        "source_company": "CZ01",
        "record_origin": "LEGACY_FILE",
        "source_priority": 2,
        "table_name": bronze_table(
            "legacy_czech_erp_customers"
        ),
    },
}


REQUIRED_SILVER_TABLES = {
    "DIM_COMPANY": silver_table(
        "dim_company"
    ),
}


# -----------------------------------------------------------------------------
# 4. Explicit schema for availability results
#
# This prevents CANNOT_DETERMINE_TYPE errors when some fields contain None.
# -----------------------------------------------------------------------------

source_check_schema = StructType([
    StructField(
        "object_type",
        StringType(),
        False,
    ),
    StructField(
        "source_name",
        StringType(),
        False,
    ),
    StructField(
        "source_system",
        StringType(),
        True,
    ),
    StructField(
        "source_company",
        StringType(),
        True,
    ),
    StructField(
        "record_origin",
        StringType(),
        True,
    ),
    StructField(
        "source_priority",
        IntegerType(),
        True,
    ),
    StructField(
        "table_name",
        StringType(),
        False,
    ),
    StructField(
        "availability_status",
        StringType(),
        False,
    ),
    StructField(
        "error_message",
        StringType(),
        True,
    ),
])


# -----------------------------------------------------------------------------
# 5. Validate customer source-table availability
# -----------------------------------------------------------------------------

source_check_results = []


for source_name, config in CUSTOMER_SOURCE_TABLES.items():
    table_name = config[
        "table_name"
    ]

    try:
        test_df = spark.sql(
            f"SELECT * FROM {table_name} LIMIT 1"
        )

        # Force Spark to resolve the object.
        test_df.collect()

        source_check_results.append({
            "object_type": "CUSTOMER_SOURCE",
            "source_name": str(
                source_name
            ),
            "source_system": str(
                config["source_system"]
            ),
            "source_company": str(
                config["source_company"]
            ),
            "record_origin": str(
                config["record_origin"]
            ),
            "source_priority": int(
                config["source_priority"]
            ),
            "table_name": str(
                table_name.replace(
                    "`",
                    "",
                )
            ),
            "availability_status": "AVAILABLE",
            "error_message": None,
        })

    except Exception as exc:
        source_check_results.append({
            "object_type": "CUSTOMER_SOURCE",
            "source_name": str(
                source_name
            ),
            "source_system": str(
                config["source_system"]
            ),
            "source_company": str(
                config["source_company"]
            ),
            "record_origin": str(
                config["record_origin"]
            ),
            "source_priority": int(
                config["source_priority"]
            ),
            "table_name": str(
                table_name.replace(
                    "`",
                    "",
                )
            ),
            "availability_status": "MISSING",
            "error_message": str(
                exc
            )[:2000],
        })


# -----------------------------------------------------------------------------
# 6. Validate required Silver dependencies
# -----------------------------------------------------------------------------

for object_name, table_name in REQUIRED_SILVER_TABLES.items():
    try:
        test_df = spark.sql(
            f"SELECT * FROM {table_name} LIMIT 1"
        )

        test_df.collect()

        source_check_results.append({
            "object_type": "SILVER_DEPENDENCY",
            "source_name": str(
                object_name
            ),
            "source_system": None,
            "source_company": None,
            "record_origin": None,
            "source_priority": None,
            "table_name": str(
                table_name.replace(
                    "`",
                    "",
                )
            ),
            "availability_status": "AVAILABLE",
            "error_message": None,
        })

    except Exception as exc:
        source_check_results.append({
            "object_type": "SILVER_DEPENDENCY",
            "source_name": str(
                object_name
            ),
            "source_system": None,
            "source_company": None,
            "record_origin": None,
            "source_priority": None,
            "table_name": str(
                table_name.replace(
                    "`",
                    "",
                )
            ),
            "availability_status": "MISSING",
            "error_message": str(
                exc
            )[:2000],
        })


# -----------------------------------------------------------------------------
# 7. Create availability DataFrame using explicit schema
# -----------------------------------------------------------------------------

SOURCE_CHECK_DF = spark.createDataFrame(
    source_check_results,
    schema=source_check_schema,
)


display(
    SOURCE_CHECK_DF.orderBy(
        "object_type",
        "source_system",
        "record_origin",
        "source_name",
    )
)


# -----------------------------------------------------------------------------
# 8. Validation summary
# -----------------------------------------------------------------------------

customer_source_count = len(
    CUSTOMER_SOURCE_TABLES
)


available_customer_source_count = (
    SOURCE_CHECK_DF
    .filter(
        (F.col("object_type") == "CUSTOMER_SOURCE")
        &
        (
            F.col("availability_status")
            == "AVAILABLE"
        )
    )
    .count()
)


missing_customer_source_count = (
    SOURCE_CHECK_DF
    .filter(
        (F.col("object_type") == "CUSTOMER_SOURCE")
        &
        (
            F.col("availability_status")
            == "MISSING"
        )
    )
    .count()
)


available_dependency_count = (
    SOURCE_CHECK_DF
    .filter(
        (F.col("object_type") == "SILVER_DEPENDENCY")
        &
        (
            F.col("availability_status")
            == "AVAILABLE"
        )
    )
    .count()
)


missing_dependency_count = (
    SOURCE_CHECK_DF
    .filter(
        (F.col("object_type") == "SILVER_DEPENDENCY")
        &
        (
            F.col("availability_status")
            == "MISSING"
        )
    )
    .count()
)


print("=" * 80)
print("DIM_CUSTOMER SOURCE CONFIGURATION")
print("=" * 80)
print(
    f"Pipeline run ID             : "
    f"{PIPELINE_RUN_ID}"
)
print(
    f"Run started UTC             : "
    f"{DIM_CUSTOMER_RUN_STARTED_UTC}"
)
print(
    f"Bronze source               : "
    f"{BRONZE_LAKEHOUSE}.{BRONZE_SCHEMA}"
)
print(
    f"Silver target               : "
    f"{SILVER_LAKEHOUSE}.{SILVER_SCHEMA}"
)
print(
    f"Customer sources expected   : "
    f"{customer_source_count}"
)
print(
    f"Customer sources available  : "
    f"{available_customer_source_count}"
)
print(
    f"Customer sources missing    : "
    f"{missing_customer_source_count}"
)
print(
    f"Silver dependencies available: "
    f"{available_dependency_count}"
)
print(
    f"Silver dependencies missing : "
    f"{missing_dependency_count}"
)
print("=" * 80)


# -----------------------------------------------------------------------------
# 9. Display missing objects before stopping
# -----------------------------------------------------------------------------

if missing_customer_source_count > 0:
    print(
        "Missing customer source tables:"
    )

    display(
        SOURCE_CHECK_DF
        .filter(
            (F.col("object_type") == "CUSTOMER_SOURCE")
            &
            (
                F.col("availability_status")
                == "MISSING"
            )
        )
        .select(
            "source_name",
            "source_system",
            "source_company",
            "record_origin",
            "table_name",
            "error_message",
        )
    )


if missing_dependency_count > 0:
    print(
        "Missing Silver dependency tables:"
    )

    display(
        SOURCE_CHECK_DF
        .filter(
            (F.col("object_type") == "SILVER_DEPENDENCY")
            &
            (
                F.col("availability_status")
                == "MISSING"
            )
        )
        .select(
            "source_name",
            "table_name",
            "error_message",
        )
    )


# -----------------------------------------------------------------------------
# 10. Stop execution if any required object is missing
# -----------------------------------------------------------------------------

if missing_customer_source_count > 0:
    raise RuntimeError(
        f"{missing_customer_source_count} required customer "
        "source table(s) are unavailable."
    )


if missing_dependency_count > 0:
    raise RuntimeError(
        f"{missing_dependency_count} required Silver "
        "dependency table(s) are unavailable."
    )


if (
    available_customer_source_count
    != customer_source_count
):
    raise RuntimeError(
        "The number of available customer sources does not "
        "match the configured source count."
    )


if (
    available_dependency_count
    != len(REQUIRED_SILVER_TABLES)
):
    raise RuntimeError(
        "The number of available Silver dependencies does not "
        "match the configured dependency count."
    )


print(
    "DIM_CUSTOMER SOURCE CONFIGURATION: READY"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26567439-594a-42b8-ade8-953a714e31f4)

DIM_CUSTOMER SOURCE CONFIGURATION
Pipeline run ID             : dim_customer_manual_20260812T142203
Run started UTC             : 2026-08-12 14:22:03.160059
Bronze source               : lh_global_finance_bronze.dbo
Silver target               : lh_global_finance_silver.dbo
Customer sources expected   : 6
Customer sources available  : 6
Customer sources missing    : 0
Silver dependencies available: 1
Silver dependencies missing : 0
DIM_CUSTOMER SOURCE CONFIGURATION: READY


In [2]:
# =============================================================================
# CELL 2 — PROFILE CUSTOMER SOURCE TABLES
#
# Purpose:
# - Record row counts and column counts for all customer sources.
# - Capture every column name, data type and nullability.
# - Compare live and legacy schemas before defining the Silver mapping.
#
# Read-only cell:
# - No source or target table is modified.
# =============================================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


# -----------------------------------------------------------------------------
# 1. Validate Cell 1 runtime objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "CUSTOMER_SOURCE_TABLES",
    "PIPELINE_RUN_ID",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 2 cannot run because these Cell 1 objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cell 1 first."
    )


# -----------------------------------------------------------------------------
# 2. Explicit result schemas
# -----------------------------------------------------------------------------

customer_profile_schema = StructType([
    StructField("source_name", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField("source_company", StringType(), False),
    StructField("record_origin", StringType(), False),
    StructField("source_priority", IntegerType(), False),
    StructField("table_name", StringType(), False),
    StructField("row_count", LongType(), True),
    StructField("column_count", IntegerType(), True),
    StructField("profile_status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("profiled_utc", TimestampType(), False),
])


customer_schema_inventory_schema = StructType([
    StructField("source_name", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField("source_company", StringType(), False),
    StructField("record_origin", StringType(), False),
    StructField("source_priority", IntegerType(), False),
    StructField("table_name", StringType(), False),
    StructField("column_position", IntegerType(), False),
    StructField("column_name", StringType(), False),
    StructField("data_type", StringType(), False),
    StructField("nullable", BooleanType(), False),
])


# -----------------------------------------------------------------------------
# 3. Profile each customer source
# -----------------------------------------------------------------------------

profiled_utc = datetime.now(
    timezone.utc
).replace(tzinfo=None)

customer_profile_rows = []
customer_schema_rows = []


for source_name, config in CUSTOMER_SOURCE_TABLES.items():
    table_name = config["table_name"]

    try:
        source_df = spark.sql(
            f"SELECT * FROM {table_name}"
        )

        row_count = source_df.count()
        column_count = len(source_df.columns)

        customer_profile_rows.append({
            "source_name": source_name,
            "source_system": config["source_system"],
            "source_company": config["source_company"],
            "record_origin": config["record_origin"],
            "source_priority": int(config["source_priority"]),
            "table_name": table_name.replace("`", ""),
            "row_count": int(row_count),
            "column_count": int(column_count),
            "profile_status": "SUCCEEDED",
            "error_message": None,
            "profiled_utc": profiled_utc,
        })

        for column_position, field in enumerate(
            source_df.schema.fields,
            start=1,
        ):
            customer_schema_rows.append({
                "source_name": source_name,
                "source_system": config["source_system"],
                "source_company": config["source_company"],
                "record_origin": config["record_origin"],
                "source_priority": int(config["source_priority"]),
                "table_name": table_name.replace("`", ""),
                "column_position": int(column_position),
                "column_name": field.name,
                "data_type": field.dataType.simpleString(),
                "nullable": bool(field.nullable),
            })

    except Exception as exc:
        customer_profile_rows.append({
            "source_name": source_name,
            "source_system": config["source_system"],
            "source_company": config["source_company"],
            "record_origin": config["record_origin"],
            "source_priority": int(config["source_priority"]),
            "table_name": table_name.replace("`", ""),
            "row_count": None,
            "column_count": None,
            "profile_status": "FAILED",
            "error_message": str(exc)[:2000],
            "profiled_utc": profiled_utc,
        })


# -----------------------------------------------------------------------------
# 4. Create profiling DataFrames
# -----------------------------------------------------------------------------

CUSTOMER_SOURCE_PROFILE_DF = spark.createDataFrame(
    customer_profile_rows,
    schema=customer_profile_schema,
)

CUSTOMER_SOURCE_SCHEMA_DF = spark.createDataFrame(
    customer_schema_rows,
    schema=customer_schema_inventory_schema,
)


# -----------------------------------------------------------------------------
# 5. Display source-level profiling
# -----------------------------------------------------------------------------

display(
    CUSTOMER_SOURCE_PROFILE_DF.orderBy(
        "source_system",
        "source_priority",
        "source_name",
    )
)


# -----------------------------------------------------------------------------
# 6. Display schema inventory
# -----------------------------------------------------------------------------

display(
    CUSTOMER_SOURCE_SCHEMA_DF.orderBy(
        "source_system",
        "source_priority",
        "source_name",
        "column_position",
    )
)


# -----------------------------------------------------------------------------
# 7. Source-system summary
# -----------------------------------------------------------------------------

CUSTOMER_SOURCE_SUMMARY_DF = (
    CUSTOMER_SOURCE_PROFILE_DF
    .groupBy(
        "source_system"
    )
    .agg(
        F.count("*").alias("source_table_count"),
        F.sum(
            F.when(
                F.col("record_origin") == "LEGACY_FILE",
                F.col("row_count"),
            ).otherwise(F.lit(0))
        ).alias("legacy_rows"),
        F.sum(
            F.when(
                F.col("record_origin") != "LEGACY_FILE",
                F.col("row_count"),
            ).otherwise(F.lit(0))
        ).alias("live_rows"),
        F.sum(
            F.coalesce(
                F.col("row_count"),
                F.lit(0),
            )
        ).alias("total_rows"),
    )
    .orderBy(
        "source_system"
    )
)


display(
    CUSTOMER_SOURCE_SUMMARY_DF
)


# -----------------------------------------------------------------------------
# 8. Validation summary
# -----------------------------------------------------------------------------

profile_success_count = (
    CUSTOMER_SOURCE_PROFILE_DF
    .filter(
        F.col("profile_status") == "SUCCEEDED"
    )
    .count()
)

profile_failure_count = (
    CUSTOMER_SOURCE_PROFILE_DF
    .filter(
        F.col("profile_status") == "FAILED"
    )
    .count()
)

empty_source_count = (
    CUSTOMER_SOURCE_PROFILE_DF
    .filter(
        F.col("profile_status") == "SUCCEEDED"
    )
    .filter(
        F.col("row_count") == 0
    )
    .count()
)

expected_source_count = len(
    CUSTOMER_SOURCE_TABLES
)


print("=" * 80)
print("DIM_CUSTOMER SOURCE PROFILING")
print("=" * 80)
print(f"Sources expected   : {expected_source_count}")
print(f"Sources profiled   : {profile_success_count}")
print(f"Profiling failures : {profile_failure_count}")
print(f"Empty sources      : {empty_source_count}")
print(
    "Schema columns     : "
    f"{CUSTOMER_SOURCE_SCHEMA_DF.count()}"
)
print("=" * 80)


if profile_failure_count > 0:
    display(
        CUSTOMER_SOURCE_PROFILE_DF
        .filter(
            F.col("profile_status") == "FAILED"
        )
        .select(
            "source_name",
            "source_system",
            "table_name",
            "error_message",
        )
    )

    raise RuntimeError(
        f"{profile_failure_count} customer source table(s) "
        "failed profiling."
    )


if profile_success_count != expected_source_count:
    raise RuntimeError(
        "The number of successfully profiled customer sources "
        "does not match the configured source count."
    )


print(
    "DIM_CUSTOMER SOURCE PROFILING: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f56c93f8-3fa5-45ac-95bf-14540e785af2)

SynapseWidget(Synapse.DataFrame, 78b2576c-3475-41c5-9816-b574717f0cce)

SynapseWidget(Synapse.DataFrame, 790a5fe4-3b6b-4b09-b6b6-0cd06d6dd50f)

DIM_CUSTOMER SOURCE PROFILING
Sources expected   : 6
Sources profiled   : 6
Profiling failures : 0
Empty sources      : 0
Schema columns     : 96
DIM_CUSTOMER SOURCE PROFILING: SUCCEEDED


In [3]:
# =============================================================================
# CELL 3 — INSPECT REPRESENTATIVE CUSTOMER SOURCE RECORDS
#
# Purpose:
# - Display representative rows from all six customer sources.
# - Review actual values before defining the conformed Silver mapping.
# - Identify live-versus-legacy business-key overlap.
#
# Read-only cell:
# - No source or target table is modified.
# =============================================================================

from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# 1. Validate upstream objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "CUSTOMER_SOURCE_TABLES",
    "CUSTOMER_SOURCE_PROFILE_DF",
    "CUSTOMER_SOURCE_SCHEMA_DF",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 3 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 and 2 first."
    )


# -----------------------------------------------------------------------------
# 2. Display representative records from every source
# -----------------------------------------------------------------------------

CUSTOMER_SOURCE_DATAFRAMES = {}


for source_name, config in CUSTOMER_SOURCE_TABLES.items():
    table_name = config["table_name"]

    source_df = spark.sql(
        f"SELECT * FROM {table_name}"
    )

    CUSTOMER_SOURCE_DATAFRAMES[
        source_name
    ] = source_df

    print("=" * 100)
    print(f"SOURCE NAME    : {source_name}")
    print(f"SOURCE SYSTEM  : {config['source_system']}")
    print(f"SOURCE COMPANY : {config['source_company']}")
    print(f"RECORD ORIGIN  : {config['record_origin']}")
    print(
        "TABLE          : "
        f"{table_name.replace('`', '')}"
    )
    print(f"ROW COUNT      : {source_df.count()}")
    print(f"COLUMN COUNT   : {len(source_df.columns)}")
    print("=" * 100)

    display(
        source_df.limit(10)
    )


# -----------------------------------------------------------------------------
# 3. Display QuickBooks customer payloads separately
# -----------------------------------------------------------------------------

qbo_live_df = CUSTOMER_SOURCE_DATAFRAMES[
    "QBO_ES_LIVE"
]


if "payload_json" in qbo_live_df.columns:
    qbo_payload_columns = [
        column_name
        for column_name in [
            "source_record_id",
            "document_number",
            "source_created_time",
            "source_last_updated_time",
            "payload_json",
            "_source_company",
            "_pipeline_run_id",
            "_ingested_utc",
        ]
        if column_name in qbo_live_df.columns
    ]

    print("=" * 100)
    print("QUICKBOOKS LIVE CUSTOMER PAYLOAD REVIEW")
    print("=" * 100)

    display(
        qbo_live_df
        .select(*qbo_payload_columns)
        .limit(10)
    )

else:
    print(
        "QuickBooks live customer table does not contain payload_json."
    )


# -----------------------------------------------------------------------------
# 4. Display concise schema comparison
# -----------------------------------------------------------------------------

key_column_patterns = [
    "id",
    "code",
    "name",
    "type",
    "country",
    "city",
    "currency",
    "term",
    "credit",
    "active",
    "created",
    "modified",
    "updated",
    "payload",
]


CUSTOMER_RELEVANT_SCHEMA_DF = (
    CUSTOMER_SOURCE_SCHEMA_DF
    .filter(
        F.lower(
            F.col("column_name")
        ).rlike(
            "|".join(key_column_patterns)
        )
    )
    .orderBy(
        "source_system",
        "source_priority",
        "source_name",
        "column_position",
    )
)


print("=" * 100)
print("CUSTOMER RELEVANT COLUMN COMPARISON")
print("=" * 100)

display(
    CUSTOMER_RELEVANT_SCHEMA_DF
)


# -----------------------------------------------------------------------------
# 5. Complete inspection summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("DIM_CUSTOMER SOURCE INSPECTION")
print("=" * 80)
print(
    f"Sources inspected : "
    f"{len(CUSTOMER_SOURCE_DATAFRAMES)}"
)
print(
    "QuickBooks payload: "
    + (
        "AVAILABLE"
        if "payload_json" in qbo_live_df.columns
        else "NOT AVAILABLE"
    )
)
print("Inspection status : SUCCEEDED")
print("=" * 80)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 5, Finished, Available, Finished, False)

SOURCE NAME    : SAP_UK_LIVE
SOURCE SYSTEM  : SAP_UK
SOURCE COMPANY : UK01
RECORD ORIGIN  : LIVE_SOURCE
TABLE          : Global-Finance-Analytics-DEV.lh_global_finance_bronze.dbo.sap_uk_ocrd
ROW COUNT      : 11
COLUMN COUNT   : 17


SynapseWidget(Synapse.DataFrame, d8fa3434-c404-4f5b-bc9b-ec3d993785d6)

SOURCE NAME    : QBO_ES_LIVE
SOURCE SYSTEM  : QBO_ES
SOURCE COMPANY : ES01
RECORD ORIGIN  : LIVE_API
TABLE          : Global-Finance-Analytics-DEV.lh_global_finance_bronze.dbo.qbo_es_customers
ROW COUNT      : 70
COLUMN COUNT   : 19


SynapseWidget(Synapse.DataFrame, 173580fc-467e-4196-b92a-2149e847f8de)

...

QUICKBOOKS LIVE CUSTOMER PAYLOAD REVIEW


SynapseWidget(Synapse.DataFrame, 32ce5c67-fe52-4a5a-90b6-8e69ed1a2bae)

CUSTOMER RELEVANT COLUMN COMPARISON


SynapseWidget(Synapse.DataFrame, 86341215-8082-4602-8e67-722fc1f67db2)

DIM_CUSTOMER SOURCE INSPECTION
Sources inspected : 6
QuickBooks payload: AVAILABLE
Inspection status : SUCCEEDED


In [4]:
# =============================================================================
# CELL 4 — PARSE AND FLATTEN QUICKBOOKS CUSTOMER PAYLOAD
#
# Purpose:
# - Parse the raw QuickBooks customer JSON payload.
# - Extract typed customer attributes needed for the Silver dimension.
# - Preserve source identifiers and ingestion metadata.
# - Prepare a clean QuickBooks live dataset for later standardisation.
#
# Important:
# - Bronze remains unchanged.
# - Sandbox geography and currency values are retained here as source values.
# - Controlled ES01 business mappings will be applied later in Silver.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DecimalType,
    StringType,
    StructField,
    StructType,
)


# -----------------------------------------------------------------------------
# 1. Validate required upstream object
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "CUSTOMER_SOURCE_DATAFRAMES",
    "PIPELINE_RUN_ID",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 4 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 to 3 first."
    )


if "QBO_ES_LIVE" not in CUSTOMER_SOURCE_DATAFRAMES:
    raise KeyError(
        "QBO_ES_LIVE is not available in CUSTOMER_SOURCE_DATAFRAMES."
    )


qbo_customer_raw_df = CUSTOMER_SOURCE_DATAFRAMES[
    "QBO_ES_LIVE"
]


if "payload_json" not in qbo_customer_raw_df.columns:
    raise RuntimeError(
        "The QuickBooks live customer table does not contain payload_json."
    )


# -----------------------------------------------------------------------------
# 2. Define explicit QuickBooks customer JSON schema
# -----------------------------------------------------------------------------

qbo_customer_payload_schema = StructType([
    StructField(
        "Id",
        StringType(),
        True,
    ),
    StructField(
        "SyncToken",
        StringType(),
        True,
    ),
    StructField(
        "Active",
        BooleanType(),
        True,
    ),
    StructField(
        "CompanyName",
        StringType(),
        True,
    ),
    StructField(
        "DisplayName",
        StringType(),
        True,
    ),
    StructField(
        "FullyQualifiedName",
        StringType(),
        True,
    ),
    StructField(
        "GivenName",
        StringType(),
        True,
    ),
    StructField(
        "MiddleName",
        StringType(),
        True,
    ),
    StructField(
        "FamilyName",
        StringType(),
        True,
    ),
    StructField(
        "Suffix",
        StringType(),
        True,
    ),
    StructField(
        "PrintOnCheckName",
        StringType(),
        True,
    ),
    StructField(
        "Job",
        BooleanType(),
        True,
    ),
    StructField(
        "IsProject",
        BooleanType(),
        True,
    ),
    StructField(
        "Level",
        StringType(),
        True,
    ),
    StructField(
        "Taxable",
        BooleanType(),
        True,
    ),
    StructField(
        "Balance",
        DecimalType(18, 2),
        True,
    ),
    StructField(
        "BalanceWithJobs",
        DecimalType(18, 2),
        True,
    ),
    StructField(
        "PreferredDeliveryMethod",
        StringType(),
        True,
    ),
    StructField(
        "ParentRef",
        StructType([
            StructField(
                "value",
                StringType(),
                True,
            ),
            StructField(
                "name",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "CurrencyRef",
        StructType([
            StructField(
                "value",
                StringType(),
                True,
            ),
            StructField(
                "name",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "PrimaryEmailAddr",
        StructType([
            StructField(
                "Address",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "PrimaryPhone",
        StructType([
            StructField(
                "FreeFormNumber",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "Mobile",
        StructType([
            StructField(
                "FreeFormNumber",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "Fax",
        StructType([
            StructField(
                "FreeFormNumber",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "WebAddr",
        StructType([
            StructField(
                "URI",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "BillAddr",
        StructType([
            StructField(
                "Id",
                StringType(),
                True,
            ),
            StructField(
                "Line1",
                StringType(),
                True,
            ),
            StructField(
                "Line2",
                StringType(),
                True,
            ),
            StructField(
                "Line3",
                StringType(),
                True,
            ),
            StructField(
                "City",
                StringType(),
                True,
            ),
            StructField(
                "Country",
                StringType(),
                True,
            ),
            StructField(
                "CountrySubDivisionCode",
                StringType(),
                True,
            ),
            StructField(
                "PostalCode",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "ShipAddr",
        StructType([
            StructField(
                "Id",
                StringType(),
                True,
            ),
            StructField(
                "Line1",
                StringType(),
                True,
            ),
            StructField(
                "Line2",
                StringType(),
                True,
            ),
            StructField(
                "Line3",
                StringType(),
                True,
            ),
            StructField(
                "City",
                StringType(),
                True,
            ),
            StructField(
                "Country",
                StringType(),
                True,
            ),
            StructField(
                "CountrySubDivisionCode",
                StringType(),
                True,
            ),
            StructField(
                "PostalCode",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "MetaData",
        StructType([
            StructField(
                "CreateTime",
                StringType(),
                True,
            ),
            StructField(
                "LastUpdatedTime",
                StringType(),
                True,
            ),
        ]),
        True,
    ),
    StructField(
        "domain",
        StringType(),
        True,
    ),
    StructField(
        "sparse",
        BooleanType(),
        True,
    ),
])


# -----------------------------------------------------------------------------
# 3. Parse and flatten the QuickBooks customer payload
# -----------------------------------------------------------------------------

QBO_CUSTOMER_PARSED_DF = (
    qbo_customer_raw_df

    .withColumn(
        "parsed_payload",
        F.from_json(
            F.col("payload_json"),
            qbo_customer_payload_schema,
        ),
    )

    .select(
        # Source identifiers
        F.col("source_record_id").cast("string").alias(
            "source_record_id"
        ),
        F.col("sync_token").cast("string").alias(
            "bronze_sync_token"
        ),
        F.col("source_created_time").cast("string").alias(
            "bronze_source_created_time"
        ),
        F.col("source_last_updated_time").cast("string").alias(
            "bronze_source_last_updated_time"
        ),
        F.col("document_number").cast("string").alias(
            "bronze_document_number"
        ),
        F.col("currency_code").cast("string").alias(
            "bronze_currency_code"
        ),
        F.col("payload_json").cast("string").alias(
            "payload_json"
        ),

        # Customer identity
        F.col("parsed_payload.Id").alias(
            "payload_customer_id"
        ),
        F.col("parsed_payload.SyncToken").alias(
            "payload_sync_token"
        ),
        F.col("parsed_payload.CompanyName").alias(
            "company_name"
        ),
        F.col("parsed_payload.DisplayName").alias(
            "display_name"
        ),
        F.col("parsed_payload.FullyQualifiedName").alias(
            "fully_qualified_name"
        ),
        F.col("parsed_payload.GivenName").alias(
            "given_name"
        ),
        F.col("parsed_payload.MiddleName").alias(
            "middle_name"
        ),
        F.col("parsed_payload.FamilyName").alias(
            "family_name"
        ),
        F.col("parsed_payload.Suffix").alias(
            "suffix"
        ),
        F.col("parsed_payload.PrintOnCheckName").alias(
            "print_on_check_name"
        ),

        # Status and classification
        F.col("parsed_payload.Active").cast("boolean").alias(
            "is_active"
        ),
        F.col("parsed_payload.Job").cast("boolean").alias(
            "is_job"
        ),
        F.col("parsed_payload.IsProject").cast("boolean").alias(
            "is_project"
        ),
        F.col("parsed_payload.Level").cast("string").alias(
            "customer_level"
        ),
        F.col("parsed_payload.Taxable").cast("boolean").alias(
            "is_taxable"
        ),
        F.col("parsed_payload.PreferredDeliveryMethod").alias(
            "preferred_delivery_method"
        ),

        # Parent relationship
        F.col("parsed_payload.ParentRef.value").alias(
            "parent_customer_id"
        ),
        F.col("parsed_payload.ParentRef.name").alias(
            "parent_customer_name"
        ),

        # Financial attributes
        F.col("parsed_payload.Balance").cast(
            DecimalType(18, 2)
        ).alias(
            "customer_balance"
        ),
        F.col("parsed_payload.BalanceWithJobs").cast(
            DecimalType(18, 2)
        ).alias(
            "customer_balance_with_jobs"
        ),
        F.col("parsed_payload.CurrencyRef.value").alias(
            "source_currency_code"
        ),
        F.col("parsed_payload.CurrencyRef.name").alias(
            "source_currency_name"
        ),

        # Contact information
        F.col("parsed_payload.PrimaryEmailAddr.Address").alias(
            "email_address"
        ),
        F.col("parsed_payload.PrimaryPhone.FreeFormNumber").alias(
            "primary_phone"
        ),
        F.col("parsed_payload.Mobile.FreeFormNumber").alias(
            "mobile_phone"
        ),
        F.col("parsed_payload.Fax.FreeFormNumber").alias(
            "fax_number"
        ),
        F.col("parsed_payload.WebAddr.URI").alias(
            "website_url"
        ),

        # Billing address
        F.col("parsed_payload.BillAddr.Line1").alias(
            "billing_address_line_1"
        ),
        F.col("parsed_payload.BillAddr.Line2").alias(
            "billing_address_line_2"
        ),
        F.col("parsed_payload.BillAddr.Line3").alias(
            "billing_address_line_3"
        ),
        F.col("parsed_payload.BillAddr.City").alias(
            "billing_city"
        ),
        F.col("parsed_payload.BillAddr.Country").alias(
            "billing_country"
        ),
        F.col(
            "parsed_payload.BillAddr.CountrySubDivisionCode"
        ).alias(
            "billing_region_code"
        ),
        F.col("parsed_payload.BillAddr.PostalCode").alias(
            "billing_postal_code"
        ),

        # Shipping address
        F.col("parsed_payload.ShipAddr.Line1").alias(
            "shipping_address_line_1"
        ),
        F.col("parsed_payload.ShipAddr.Line2").alias(
            "shipping_address_line_2"
        ),
        F.col("parsed_payload.ShipAddr.Line3").alias(
            "shipping_address_line_3"
        ),
        F.col("parsed_payload.ShipAddr.City").alias(
            "shipping_city"
        ),
        F.col("parsed_payload.ShipAddr.Country").alias(
            "shipping_country"
        ),
        F.col(
            "parsed_payload.ShipAddr.CountrySubDivisionCode"
        ).alias(
            "shipping_region_code"
        ),
        F.col("parsed_payload.ShipAddr.PostalCode").alias(
            "shipping_postal_code"
        ),

        # Source timestamps
        F.to_timestamp(
            F.col("parsed_payload.MetaData.CreateTime"),
            "yyyy-MM-dd'T'HH:mm:ssXXX",
        ).alias(
            "source_created_utc"
        ),
        F.to_timestamp(
            F.col("parsed_payload.MetaData.LastUpdatedTime"),
            "yyyy-MM-dd'T'HH:mm:ssXXX",
        ).alias(
            "source_modified_utc"
        ),

        # Bronze audit metadata
        F.col("_source_system").cast("string").alias(
            "bronze_source_system"
        ),
        F.col("_source_company").cast("string").alias(
            "bronze_source_company"
        ),
        F.col("_source_environment").cast("string").alias(
            "bronze_source_environment"
        ),
        F.col("_pipeline_run_id").cast("string").alias(
            "bronze_pipeline_run_id"
        ),
        F.col("_ingested_utc").cast("timestamp").alias(
            "bronze_ingested_utc"
        ),
    )
)


# -----------------------------------------------------------------------------
# 4. Add conformed helper attributes
# -----------------------------------------------------------------------------

QBO_CUSTOMER_PARSED_DF = (
    QBO_CUSTOMER_PARSED_DF

    .withColumn(
        "resolved_customer_id",
        F.coalesce(
            F.col("payload_customer_id"),
            F.col("source_record_id"),
        ),
    )

    .withColumn(
        "resolved_customer_name",
        F.coalesce(
            F.col("company_name"),
            F.col("display_name"),
            F.col("fully_qualified_name"),
            F.col("print_on_check_name"),
            F.concat_ws(
                " ",
                F.col("given_name"),
                F.col("family_name"),
            ),
            F.col("bronze_document_number"),
        ),
    )

    .withColumn(
        "resolved_sync_token",
        F.coalesce(
            F.col("payload_sync_token"),
            F.col("bronze_sync_token"),
        ),
    )

    .withColumn(
        "resolved_currency_code",
        F.coalesce(
            F.col("source_currency_code"),
            F.col("bronze_currency_code"),
        ),
    )

    .withColumn(
        "resolved_city",
        F.coalesce(
            F.col("billing_city"),
            F.col("shipping_city"),
        ),
    )

    .withColumn(
        "resolved_postal_code",
        F.coalesce(
            F.col("billing_postal_code"),
            F.col("shipping_postal_code"),
        ),
    )

    .withColumn(
        "resolved_country",
        F.coalesce(
            F.col("billing_country"),
            F.col("shipping_country"),
        ),
    )
)


# -----------------------------------------------------------------------------
# 5. Data-quality validation
# -----------------------------------------------------------------------------

qbo_customer_row_count = (
    QBO_CUSTOMER_PARSED_DF.count()
)

invalid_id_count = (
    QBO_CUSTOMER_PARSED_DF
    .filter(
        F.col("resolved_customer_id").isNull()
        | (
            F.trim(
                F.col("resolved_customer_id")
            )
            == ""
        )
    )
    .count()
)

invalid_name_count = (
    QBO_CUSTOMER_PARSED_DF
    .filter(
        F.col("resolved_customer_name").isNull()
        | (
            F.trim(
                F.col("resolved_customer_name")
            )
            == ""
        )
    )
    .count()
)

unparsed_payload_count = (
    qbo_customer_raw_df
    .withColumn(
        "parsed_payload",
        F.from_json(
            F.col("payload_json"),
            qbo_customer_payload_schema,
        ),
    )
    .filter(
        F.col("payload_json").isNotNull()
        & F.col("parsed_payload").isNull()
    )
    .count()
)

duplicate_customer_id_count = (
    QBO_CUSTOMER_PARSED_DF
    .groupBy(
        "resolved_customer_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


print("=" * 80)
print("QUICKBOOKS CUSTOMER PAYLOAD PARSING")
print("=" * 80)
print(f"Rows parsed              : {qbo_customer_row_count}")
print(f"Invalid customer IDs     : {invalid_id_count}")
print(f"Invalid customer names   : {invalid_name_count}")
print(f"Unparsed JSON payloads   : {unparsed_payload_count}")
print(f"Duplicate customer IDs   : {duplicate_customer_id_count}")
print("=" * 80)


if invalid_id_count > 0:
    display(
        QBO_CUSTOMER_PARSED_DF
        .filter(
            F.col("resolved_customer_id").isNull()
            | (
                F.trim(
                    F.col("resolved_customer_id")
                )
                == ""
            )
        )
    )

    raise RuntimeError(
        "One or more QuickBooks customer records do not have "
        "a valid customer ID."
    )


if invalid_name_count > 0:
    display(
        QBO_CUSTOMER_PARSED_DF
        .filter(
            F.col("resolved_customer_name").isNull()
            | (
                F.trim(
                    F.col("resolved_customer_name")
                )
                == ""
            )
        )
    )

    raise RuntimeError(
        "One or more QuickBooks customer records do not have "
        "a valid customer name."
    )


if unparsed_payload_count > 0:
    raise RuntimeError(
        f"{unparsed_payload_count} QuickBooks customer JSON "
        "payload(s) could not be parsed."
    )


if duplicate_customer_id_count > 0:
    display(
        QBO_CUSTOMER_PARSED_DF
        .groupBy(
            "resolved_customer_id"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
    )

    raise RuntimeError(
        "Duplicate QuickBooks customer IDs were found."
    )


# -----------------------------------------------------------------------------
# 6. Display safe parsed output
# -----------------------------------------------------------------------------

display(
    QBO_CUSTOMER_PARSED_DF
    .select(
        "resolved_customer_id",
        "resolved_customer_name",
        "company_name",
        "display_name",
        "is_active",
        "is_job",
        "parent_customer_id",
        "resolved_currency_code",
        "customer_balance",
        "email_address",
        "primary_phone",
        "resolved_city",
        "resolved_postal_code",
        "source_created_utc",
        "source_modified_utc",
    )
    .orderBy(
        "resolved_customer_id"
    )
)


print(
    "QUICKBOOKS CUSTOMER PAYLOAD PARSING: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 6, Finished, Available, Finished, False)

QUICKBOOKS CUSTOMER PAYLOAD PARSING
Rows parsed              : 70
Invalid customer IDs     : 0
Invalid customer names   : 0
Unparsed JSON payloads   : 0
Duplicate customer IDs   : 0


SynapseWidget(Synapse.DataFrame, d76108b4-7307-48c7-bb91-9e460d3771fc)

QUICKBOOKS CUSTOMER PAYLOAD PARSING: SUCCEEDED


In [5]:
# =============================================================================
# CELL 5 — STANDARDISE ALL CUSTOMER SOURCES
#
# Purpose:
# - Map all six customer sources into one common Silver customer contract.
# - Preserve source-system lineage.
# - Prepare records for live-over-legacy deduplication.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


# -----------------------------------------------------------------------------
# 1. Validate upstream objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "CUSTOMER_SOURCE_DATAFRAMES",
    "QBO_CUSTOMER_PARSED_DF",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 5 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 to 4 first."
    )


# -----------------------------------------------------------------------------
# 2. SAP UK live
# -----------------------------------------------------------------------------

sap_live_df = CUSTOMER_SOURCE_DATAFRAMES[
    "SAP_UK_LIVE"
]


SAP_CUSTOMER_STANDARDISED_DF = (
    sap_live_df
    .filter(
        F.upper(
            F.trim(
                F.col("CardType")
            )
        ) == F.lit("C")
    )
    .select(
        F.lit("SAP_UK").alias(
            "source_system"
        ),
        F.lit("UK01").alias(
            "source_company"
        ),
        F.lit("LIVE_SOURCE").alias(
            "record_origin"
        ),
        F.lit(1).cast("int").alias(
            "source_priority"
        ),

        F.col("CardCode").cast("string").alias(
            "customer_id"
        ),
        F.col("CardName").cast("string").alias(
            "customer_name"
        ),
        F.lit("Business").cast("string").alias(
            "customer_type"
        ),

        F.col("Country").cast("string").alias(
            "country_code"
        ),
        F.col("City").cast("string").alias(
            "city"
        ),
        F.col("PostCode").cast("string").alias(
            "postal_code"
        ),

        F.col("Currency").cast("string").alias(
            "currency_code"
        ),

        F.lit(None).cast("string").alias(
            "payment_terms"
        ),

        F.col("CreditLine")
        .cast(DecimalType(18, 2))
        .alias(
            "credit_limit"
        ),

        F.col("Balance")
        .cast(DecimalType(18, 2))
        .alias(
            "current_balance"
        ),

        (
            F.upper(
                F.trim(
                    F.col("ValidFor").cast("string")
                )
            ) == F.lit("Y")
        ).alias(
            "is_active"
        ),

        F.col("Phone1").cast("string").alias(
            "primary_phone"
        ),

        F.lit(None).cast("string").alias(
            "email_address"
        ),

        F.lit(None).cast("string").alias(
            "parent_customer_id"
        ),

        F.lit(False).cast("boolean").alias(
            "is_job"
        ),

        F.col("CreateDate")
        .cast("timestamp")
        .alias(
            "source_created_utc"
        ),

        F.coalesce(
            F.col("UpdateDate").cast("timestamp"),
            F.col("LastModifiedTS").cast("timestamp"),
        ).alias(
            "source_modified_utc"
        ),
    )
)


# -----------------------------------------------------------------------------
# 3. QuickBooks live
# -----------------------------------------------------------------------------

QBO_CUSTOMER_STANDARDISED_DF = (
    QBO_CUSTOMER_PARSED_DF
    .select(
        F.lit("QBO_ES").alias(
            "source_system"
        ),
        F.lit("ES01").alias(
            "source_company"
        ),
        F.lit("LIVE_API").alias(
            "record_origin"
        ),
        F.lit(1).cast("int").alias(
            "source_priority"
        ),

        F.col("resolved_customer_id").cast("string").alias(
            "customer_id"
        ),

        F.trim(
            F.col("resolved_customer_name")
        ).alias(
            "customer_name"
        ),

        F.when(
            F.col("is_job") == True,
            F.lit("Job")
        )
        .when(
            F.col("company_name").isNotNull(),
            F.lit("Business")
        )
        .otherwise(
            F.lit("Individual")
        )
        .alias(
            "customer_type"
        ),

        F.coalesce(
            F.when(
                F.upper(
                    F.trim(
                        F.col("resolved_country")
                    )
                ).isin("USA", "UNITED STATES"),
                F.lit("US")
            ),
            F.when(
                F.col("resolved_customer_name").rlike(".*\\bSL\\b.*"),
                F.lit("ES")
            ),
            F.when(
                F.col("resolved_customer_name").rlike(".*\\bSA\\b.*"),
                F.lit("ES")
            ),
            F.when(
                F.col("resolved_customer_name").rlike(".*\\bLDA\\b.*"),
                F.lit("PT")
            ),
            F.when(
                F.col("resolved_customer_name").rlike(".*\\bGMBH\\b.*"),
                F.lit("DE")
            ),
            F.when(
                F.col("resolved_customer_name").rlike(".*\\bSAS\\b.*"),
                F.lit("FR")
            ),
            F.lit(None).cast("string"),
        ).alias(
            "country_code"
        ),

        F.col("resolved_city").cast("string").alias(
            "city"
        ),

        F.col("resolved_postal_code").cast("string").alias(
            "postal_code"
        ),

        F.col("resolved_currency_code").cast("string").alias(
            "currency_code"
        ),

        F.lit(None).cast("string").alias(
            "payment_terms"
        ),

        F.lit(None)
        .cast(DecimalType(18, 2))
        .alias(
            "credit_limit"
        ),

        F.col("customer_balance")
        .cast(DecimalType(18, 2))
        .alias(
            "current_balance"
        ),

        F.coalesce(
            F.col("is_active"),
            F.lit(True),
        ).alias(
            "is_active"
        ),

        F.col("primary_phone").cast("string").alias(
            "primary_phone"
        ),

        F.col("email_address").cast("string").alias(
            "email_address"
        ),

        F.col("parent_customer_id").cast("string").alias(
            "parent_customer_id"
        ),

        F.coalesce(
            F.col("is_job"),
            F.lit(False),
        ).alias(
            "is_job"
        ),

        F.col("source_created_utc").cast("timestamp").alias(
            "source_created_utc"
        ),

        F.col("source_modified_utc").cast("timestamp").alias(
            "source_modified_utc"
        ),
    )
)


# -----------------------------------------------------------------------------
# 4. Czech ERP live
# -----------------------------------------------------------------------------

czech_live_df = CUSTOMER_SOURCE_DATAFRAMES[
    "CZECH_ERP_LIVE"
]


CZECH_CUSTOMER_STANDARDISED_DF = (
    czech_live_df
    .select(
        F.lit("CZECH_ERP").alias(
            "source_system"
        ),
        F.lit("CZ01").alias(
            "source_company"
        ),
        F.lit("LIVE_SOURCE").alias(
            "record_origin"
        ),
        F.lit(1).cast("int").alias(
            "source_priority"
        ),

        F.col("CustomerCode").cast("string").alias(
            "customer_id"
        ),
        F.col("CustomerName").cast("string").alias(
            "customer_name"
        ),
        F.col("CustomerType").cast("string").alias(
            "customer_type"
        ),

        F.col("CountryCode").cast("string").alias(
            "country_code"
        ),
        F.col("City").cast("string").alias(
            "city"
        ),
        F.lit(None).cast("string").alias(
            "postal_code"
        ),

        F.lit("CZK").cast("string").alias(
            "currency_code"
        ),

        F.concat(
            F.lit("NET"),
            F.col("PaymentTermsDays").cast("string"),
        ).alias(
            "payment_terms"
        ),

        F.col("CreditLimit")
        .cast(DecimalType(18, 2))
        .alias(
            "credit_limit"
        ),

        F.lit(None)
        .cast(DecimalType(18, 2))
        .alias(
            "current_balance"
        ),

        F.col("IsActive").cast("boolean").alias(
            "is_active"
        ),

        F.lit(None).cast("string").alias(
            "primary_phone"
        ),

        F.lit(None).cast("string").alias(
            "email_address"
        ),

        F.lit(None).cast("string").alias(
            "parent_customer_id"
        ),

        F.lit(False).cast("boolean").alias(
            "is_job"
        ),

        F.col("CreatedDate")
        .cast("timestamp")
        .alias(
            "source_created_utc"
        ),

        F.col("LastModifiedDate")
        .cast("timestamp")
        .alias(
            "source_modified_utc"
        ),
    )
)


# -----------------------------------------------------------------------------
# 5. Reusable legacy standardisation
# -----------------------------------------------------------------------------

def standardise_legacy_customer(
    source_df,
    source_system: str,
    source_company: str,
):
    return (
        source_df
        .select(
            F.lit(source_system).alias(
                "source_system"
            ),
            F.lit(source_company).alias(
                "source_company"
            ),
            F.lit("LEGACY_FILE").alias(
                "record_origin"
            ),
            F.lit(2).cast("int").alias(
                "source_priority"
            ),

            F.col("customer_id").cast("string").alias(
                "customer_id"
            ),
            F.col("customer_name").cast("string").alias(
                "customer_name"
            ),
            F.col("customer_type").cast("string").alias(
                "customer_type"
            ),

            F.col("country_code").cast("string").alias(
                "country_code"
            ),
            F.col("city").cast("string").alias(
                "city"
            ),
            F.lit(None).cast("string").alias(
                "postal_code"
            ),

            F.col("currency_code").cast("string").alias(
                "currency_code"
            ),
            F.col("payment_terms").cast("string").alias(
                "payment_terms"
            ),

            F.col("credit_limit")
            .cast(DecimalType(18, 2))
            .alias(
                "credit_limit"
            ),

            F.lit(None)
            .cast(DecimalType(18, 2))
            .alias(
                "current_balance"
            ),

            F.col("is_active").cast("boolean").alias(
                "is_active"
            ),

            F.lit(None).cast("string").alias(
                "primary_phone"
            ),

            F.lit(None).cast("string").alias(
                "email_address"
            ),

            F.lit(None).cast("string").alias(
                "parent_customer_id"
            ),

            F.lit(False).cast("boolean").alias(
                "is_job"
            ),

            F.col("created_date")
            .cast("timestamp")
            .alias(
                "source_created_utc"
            ),

            F.col("last_modified_date")
            .cast("timestamp")
            .alias(
                "source_modified_utc"
            ),
        )
    )


SAP_LEGACY_STANDARDISED_DF = standardise_legacy_customer(
    CUSTOMER_SOURCE_DATAFRAMES[
        "SAP_UK_LEGACY"
    ],
    "SAP_UK",
    "UK01",
)


QBO_LEGACY_STANDARDISED_DF = standardise_legacy_customer(
    CUSTOMER_SOURCE_DATAFRAMES[
        "QBO_ES_LEGACY"
    ],
    "QBO_ES",
    "ES01",
)


CZECH_LEGACY_STANDARDISED_DF = standardise_legacy_customer(
    CUSTOMER_SOURCE_DATAFRAMES[
        "CZECH_ERP_LEGACY"
    ],
    "CZECH_ERP",
    "CZ01",
)


# -----------------------------------------------------------------------------
# 6. Union all six sources
# -----------------------------------------------------------------------------

CUSTOMER_STANDARDISED_ALL_DF = (
    SAP_CUSTOMER_STANDARDISED_DF
    .unionByName(
        QBO_CUSTOMER_STANDARDISED_DF
    )
    .unionByName(
        CZECH_CUSTOMER_STANDARDISED_DF
    )
    .unionByName(
        SAP_LEGACY_STANDARDISED_DF
    )
    .unionByName(
        QBO_LEGACY_STANDARDISED_DF
    )
    .unionByName(
        CZECH_LEGACY_STANDARDISED_DF
    )
)


# -----------------------------------------------------------------------------
# 7. Final string standardisation
# -----------------------------------------------------------------------------

CUSTOMER_STANDARDISED_ALL_DF = (
    CUSTOMER_STANDARDISED_ALL_DF
    .withColumn(
        "source_system",
        F.upper(
            F.trim(
                F.col("source_system")
            )
        )
    )
    .withColumn(
        "source_company",
        F.upper(
            F.trim(
                F.col("source_company")
            )
        )
    )
    .withColumn(
        "customer_id",
        F.trim(
            F.col("customer_id")
        )
    )
    .withColumn(
        "customer_name",
        F.trim(
            F.col("customer_name")
        )
    )
    .withColumn(
        "customer_type",
        F.trim(
            F.col("customer_type")
        )
    )
    .withColumn(
        "country_code",
        F.upper(
            F.trim(
                F.col("country_code")
            )
        )
    )
    .withColumn(
        "city",
        F.trim(
            F.col("city")
        )
    )
    .withColumn(
        "postal_code",
        F.trim(
            F.col("postal_code")
        )
    )
    .withColumn(
        "currency_code",
        F.upper(
            F.trim(
                F.col("currency_code")
            )
        )
    )
    .withColumn(
        "payment_terms",
        F.upper(
            F.trim(
                F.col("payment_terms")
            )
        )
    )
)


# -----------------------------------------------------------------------------
# 8. Validate row reconciliation
# -----------------------------------------------------------------------------

expected_customer_rows = (
    SAP_CUSTOMER_STANDARDISED_DF.count()
    + QBO_CUSTOMER_STANDARDISED_DF.count()
    + CZECH_CUSTOMER_STANDARDISED_DF.count()
    + SAP_LEGACY_STANDARDISED_DF.count()
    + QBO_LEGACY_STANDARDISED_DF.count()
    + CZECH_LEGACY_STANDARDISED_DF.count()
)

actual_customer_rows = (
    CUSTOMER_STANDARDISED_ALL_DF.count()
)


print("=" * 80)
print("DIM_CUSTOMER SOURCE STANDARDISATION")
print("=" * 80)
print(f"Expected rows : {expected_customer_rows}")
print(f"Actual rows   : {actual_customer_rows}")
print("=" * 80)


if actual_customer_rows != expected_customer_rows:
    raise RuntimeError(
        "Customer standardisation row reconciliation failed."
    )


display(
    CUSTOMER_STANDARDISED_ALL_DF
    .groupBy(
        "source_system",
        "record_origin",
    )
    .agg(
        F.count("*").alias(
            "row_count"
        )
    )
    .orderBy(
        "source_system",
        "record_origin",
    )
)


display(
    CUSTOMER_STANDARDISED_ALL_DF.limit(50)
)


print(
    "DIM_CUSTOMER SOURCE STANDARDISATION: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 7, Finished, Available, Finished, False)

DIM_CUSTOMER SOURCE STANDARDISATION
Expected rows : 981
Actual rows   : 981


SynapseWidget(Synapse.DataFrame, 35dd6044-0e2a-4a88-b6a5-6f3cc1acdaf8)

SynapseWidget(Synapse.DataFrame, 752a6025-5537-420e-b987-849d65938dc0)

DIM_CUSTOMER SOURCE STANDARDISATION: SUCCEEDED


In [7]:
# =============================================================================
# CELL 6 — CUSTOMER BUSINESS KEYS AND LIVE-OVER-LEGACY DEDUPLICATION
#
# Purpose:
# - Create a stable source-system customer business key.
# - Remove duplicate source records.
# - Prioritise live/API records over legacy-file records.
# - Retain the most recently modified record within the same priority.
#
# Priority:
# 1 = LIVE_SOURCE / LIVE_API
# 2 = LEGACY_FILE
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import Window


# -----------------------------------------------------------------------------
# 1. Validate required upstream object
# -----------------------------------------------------------------------------

if "CUSTOMER_STANDARDISED_ALL_DF" not in globals():
    raise NameError(
        "CUSTOMER_STANDARDISED_ALL_DF is unavailable. "
        "Run Cells 1 through 5 first."
    )


# -----------------------------------------------------------------------------
# 2. Create stable customer business keys
#
# Business key pattern:
# SOURCE_SYSTEM | SOURCE_COMPANY | CUSTOMER_ID
#
# This avoids collisions where different ERP systems use the same local ID.
# -----------------------------------------------------------------------------

CUSTOMER_KEYED_DF = (
    CUSTOMER_STANDARDISED_ALL_DF

    .withColumn(
        "customer_business_key",
        F.concat_ws(
            "|",
            F.col("source_system"),
            F.col("source_company"),
            F.col("customer_id"),
        ),
    )

    # Timestamp used to choose the latest record within the same source priority.
    .withColumn(
        "deduplication_timestamp",
        F.coalesce(
            F.col("source_modified_utc"),
            F.col("source_created_utc"),
            F.lit("1900-01-01 00:00:00").cast("timestamp"),
        ),
    )
)


# -----------------------------------------------------------------------------
# 3. Validate business-key creation
# -----------------------------------------------------------------------------

null_business_key_count = (
    CUSTOMER_KEYED_DF
    .filter(
        F.col("customer_business_key").isNull()
        | (
            F.trim(
                F.col("customer_business_key")
            )
            == ""
        )
    )
    .count()
)


if null_business_key_count > 0:
    raise RuntimeError(
        f"{null_business_key_count} customer records have "
        "an invalid business key."
    )


# -----------------------------------------------------------------------------
# 4. Identify duplicates before deduplication
# -----------------------------------------------------------------------------

duplicate_business_key_df = (
    CUSTOMER_KEYED_DF
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_business_key_count = (
    duplicate_business_key_df.count()
)


duplicate_record_count = (
    duplicate_business_key_df
    .agg(
        F.coalesce(
            F.sum(
                F.col("count") - F.lit(1)
            ),
            F.lit(0),
        ).alias(
            "duplicate_record_count"
        )
    )
    .first()[
        "duplicate_record_count"
    ]
)


# -----------------------------------------------------------------------------
# 5. Rank records by source priority and recency
#
# Lower source_priority wins:
# - live/API = 1
# - legacy = 2
#
# If two records share the same priority, the latest modified record wins.
# -----------------------------------------------------------------------------

customer_deduplication_window = (
    Window
    .partitionBy(
        "customer_business_key"
    )
    .orderBy(
        F.col("source_priority").asc(),
        F.col("deduplication_timestamp").desc(),
        F.col("record_origin").asc(),
    )
)


CUSTOMER_RANKED_DF = (
    CUSTOMER_KEYED_DF
    .withColumn(
        "deduplication_rank",
        F.row_number().over(
            customer_deduplication_window
        ),
    )
)


CUSTOMER_DEDUPLICATED_DF = (
    CUSTOMER_RANKED_DF
    .filter(
        F.col("deduplication_rank") == 1
    )
    .drop(
        "deduplication_rank",
        "deduplication_timestamp",
    )
)


CUSTOMER_DUPLICATE_REJECTS_DF = (
    CUSTOMER_RANKED_DF
    .filter(
        F.col("deduplication_rank") > 1
    )
    .withColumn(
        "rejection_reason",
        F.lit(
            "DUPLICATE_BUSINESS_KEY_LOWER_PRIORITY_OR_OLDER_RECORD"
        ),
    )
)


# -----------------------------------------------------------------------------
# 6. Reconcile input and output counts
# -----------------------------------------------------------------------------

input_customer_count = (
    CUSTOMER_STANDARDISED_ALL_DF.count()
)

deduplicated_customer_count = (
    CUSTOMER_DEDUPLICATED_DF.count()
)

rejected_duplicate_count = (
    CUSTOMER_DUPLICATE_REJECTS_DF.count()
)


if (
    deduplicated_customer_count
    + rejected_duplicate_count
    != input_customer_count
):
    raise RuntimeError(
        "Customer deduplication row reconciliation failed. "
        f"Input={input_customer_count}; "
        f"retained={deduplicated_customer_count}; "
        f"rejected={rejected_duplicate_count}."
    )


# -----------------------------------------------------------------------------
# 7. Confirm no duplicate business keys remain
# -----------------------------------------------------------------------------

remaining_duplicate_count = (
    CUSTOMER_DEDUPLICATED_DF
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if remaining_duplicate_count > 0:
    raise RuntimeError(
        "Duplicate customer business keys remain after deduplication."
    )


# -----------------------------------------------------------------------------
# 8. Display summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("DIM_CUSTOMER BUSINESS KEY AND DEDUPLICATION")
print("=" * 80)
print(f"Input rows                  : {input_customer_count}")
print(f"Duplicate business keys     : {duplicate_business_key_count}")
print(f"Duplicate records identified: {duplicate_record_count}")
print(f"Rows retained               : {deduplicated_customer_count}")
print(f"Rows rejected               : {rejected_duplicate_count}")
print(f"Remaining duplicate keys    : {remaining_duplicate_count}")
print("=" * 80)


display(
    CUSTOMER_DEDUPLICATED_DF
    .groupBy(
        "source_system",
        "record_origin",
    )
    .agg(
        F.count("*").alias(
            "retained_rows"
        )
    )
    .orderBy(
        "source_system",
        "record_origin",
    )
)


if rejected_duplicate_count > 0:
    print("Duplicate records rejected:")

    display(
        CUSTOMER_DUPLICATE_REJECTS_DF
        .select(
            "customer_business_key",
            "customer_id",
            "customer_name",
            "source_system",
            "source_company",
            "record_origin",
            "source_priority",
            "source_created_utc",
            "source_modified_utc",
            "rejection_reason",
        )
        .orderBy(
            "customer_business_key",
            "source_priority",
        )
    )


print(
    "DIM_CUSTOMER BUSINESS KEY AND DEDUPLICATION: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 9, Finished, Available, Finished, False)

DIM_CUSTOMER BUSINESS KEY AND DEDUPLICATION
Input rows                  : 981
Duplicate business keys     : 0
Duplicate records identified: 0
Rows retained               : 981
Rows rejected               : 0
Remaining duplicate keys    : 0


SynapseWidget(Synapse.DataFrame, 1b15a199-c9b2-4ca9-8403-7964c49c1674)

DIM_CUSTOMER BUSINESS KEY AND DEDUPLICATION: SUCCEEDED


In [8]:
# =============================================================================
# CELL 7 — DIM_CUSTOMER PRE-WRITE VALIDATION AND COMPANY-KEY RESOLUTION
#
# Purpose:
# - Validate mandatory customer attributes.
# - Confirm every customer maps to a current company in dbo.dim_company.
# - Validate country, currency, email and financial values.
# - Separate critical failures from non-blocking warnings.
# - Produce CUSTOMER_VALIDATED_DF for the Silver dimension load.
#
# This cell does not write to the Silver target table.
# =============================================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
)


# -----------------------------------------------------------------------------
# 1. Validate required upstream objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "CUSTOMER_DEDUPLICATED_DF",
    "PIPELINE_RUN_ID",
    "silver_table",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 7 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 through 6 first."
    )


VALIDATION_UTC = datetime.now(
    timezone.utc
).replace(tzinfo=None)


# -----------------------------------------------------------------------------
# 2. Read the current company dimension
# -----------------------------------------------------------------------------

DIM_COMPANY_TABLE = silver_table(
    "dim_company"
)


CURRENT_COMPANY_DF = (
    spark.sql(
        f"""
        SELECT
            company_key,
            company_business_key,
            company_code,
            source_system,
            source_company,
            country_code AS company_country_code,
            base_currency_code AS company_currency_code
        FROM {DIM_COMPANY_TABLE}
        WHERE is_current = true
        """
    )
)


current_company_count = CURRENT_COMPANY_DF.count()


if current_company_count != 3:
    raise RuntimeError(
        "Expected 3 current company records in dim_company, "
        f"but found {current_company_count}."
    )


# -----------------------------------------------------------------------------
# 3. Attach the company surrogate key to every customer
# -----------------------------------------------------------------------------

CUSTOMER_WITH_COMPANY_DF = (
    CUSTOMER_DEDUPLICATED_DF.alias("customer")
    .join(
        CURRENT_COMPANY_DF.alias("company"),
        on=[
            F.col("customer.source_system")
            == F.col("company.source_system"),

            F.col("customer.source_company")
            == F.col("company.source_company"),
        ],
        how="left",
    )
    .select(
        F.col("customer.*"),
        F.col("company.company_key").alias(
            "company_key"
        ),
        F.col("company.company_business_key").alias(
            "company_business_key"
        ),
        F.col("company.company_code").alias(
            "company_code"
        ),
        F.col("company.company_country_code").alias(
            "company_country_code"
        ),
        F.col("company.company_currency_code").alias(
            "company_currency_code"
        ),
    )
)


# -----------------------------------------------------------------------------
# 4. Define critical validation failures
#
# Critical failures stop the load.
# -----------------------------------------------------------------------------

missing_business_key_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("customer_business_key").isNull()
        | (
            F.trim(
                F.col("customer_business_key")
            ) == ""
        )
    )
)


missing_customer_id_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("customer_id").isNull()
        | (
            F.trim(
                F.col("customer_id")
            ) == ""
        )
    )
)


missing_customer_name_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("customer_name").isNull()
        | (
            F.trim(
                F.col("customer_name")
            ) == ""
        )
    )
)


unknown_company_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("company_key").isNull()
    )
)


duplicate_business_key_df = (
    CUSTOMER_WITH_COMPANY_DF
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)


invalid_source_system_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        ~F.col("source_system").isin(
            "SAP_UK",
            "QBO_ES",
            "CZECH_ERP",
        )
    )
)


invalid_source_company_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        ~F.col("source_company").isin(
            "UK01",
            "ES01",
            "CZ01",
        )
    )
)


invalid_origin_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        ~F.col("record_origin").isin(
            "LIVE_SOURCE",
            "LIVE_API",
            "LEGACY_FILE",
        )
    )
)


negative_credit_limit_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("credit_limit") < F.lit(0)
    )
)


# -----------------------------------------------------------------------------
# 5. Define non-blocking validation warnings
#
# Warnings are recorded but do not stop the load because some source systems
# legitimately omit these optional attributes.
# -----------------------------------------------------------------------------

invalid_country_format_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("country_code").isNotNull()
        &
        ~F.upper(
            F.trim(
                F.col("country_code")
            )
        ).rlike(
            "^[A-Z]{2}$"
        )
    )
)


invalid_currency_format_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("currency_code").isNotNull()
        &
        ~F.upper(
            F.trim(
                F.col("currency_code")
            )
        ).rlike(
            "^[A-Z]{3}$"
        )
    )
)


invalid_email_format_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("email_address").isNotNull()
        &
        (
            F.trim(
                F.col("email_address")
            ) != ""
        )
        &
        ~F.col("email_address").rlike(
            r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"
        )
    )
)


missing_country_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("country_code").isNull()
        | (
            F.trim(
                F.col("country_code")
            ) == ""
        )
    )
)


missing_currency_df = (
    CUSTOMER_WITH_COMPANY_DF
    .filter(
        F.col("currency_code").isNull()
        | (
            F.trim(
                F.col("currency_code")
            ) == ""
        )
    )
)


# -----------------------------------------------------------------------------
# 6. Calculate validation counts
# -----------------------------------------------------------------------------

input_customer_count = (
    CUSTOMER_WITH_COMPANY_DF.count()
)


missing_business_key_count = (
    missing_business_key_df.count()
)

missing_customer_id_count = (
    missing_customer_id_df.count()
)

missing_customer_name_count = (
    missing_customer_name_df.count()
)

unknown_company_count = (
    unknown_company_df.count()
)

duplicate_business_key_count = (
    duplicate_business_key_df.count()
)

invalid_source_system_count = (
    invalid_source_system_df.count()
)

invalid_source_company_count = (
    invalid_source_company_df.count()
)

invalid_origin_count = (
    invalid_origin_df.count()
)

negative_credit_limit_count = (
    negative_credit_limit_df.count()
)

invalid_country_format_count = (
    invalid_country_format_df.count()
)

invalid_currency_format_count = (
    invalid_currency_format_df.count()
)

invalid_email_format_count = (
    invalid_email_format_df.count()
)

missing_country_count = (
    missing_country_df.count()
)

missing_currency_count = (
    missing_currency_df.count()
)


# -----------------------------------------------------------------------------
# 7. Build an explicit validation summary
# -----------------------------------------------------------------------------

validation_summary_schema = StructType([
    StructField(
        "validation_rule",
        StringType(),
        False,
    ),
    StructField(
        "severity",
        StringType(),
        False,
    ),
    StructField(
        "rows_checked",
        LongType(),
        False,
    ),
    StructField(
        "rows_failed",
        LongType(),
        False,
    ),
    StructField(
        "validation_status",
        StringType(),
        False,
    ),
])


validation_summary_rows = [
    {
        "validation_rule": "CUSTOMER_BUSINESS_KEY_REQUIRED",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": missing_business_key_count,
        "validation_status": (
            "PASSED"
            if missing_business_key_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "CUSTOMER_ID_REQUIRED",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": missing_customer_id_count,
        "validation_status": (
            "PASSED"
            if missing_customer_id_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "CUSTOMER_NAME_REQUIRED",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": missing_customer_name_count,
        "validation_status": (
            "PASSED"
            if missing_customer_name_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "COMPANY_KEY_RESOLUTION",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": unknown_company_count,
        "validation_status": (
            "PASSED"
            if unknown_company_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "UNIQUE_CUSTOMER_BUSINESS_KEY",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": duplicate_business_key_count,
        "validation_status": (
            "PASSED"
            if duplicate_business_key_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "VALID_SOURCE_SYSTEM",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": invalid_source_system_count,
        "validation_status": (
            "PASSED"
            if invalid_source_system_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "VALID_SOURCE_COMPANY",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": invalid_source_company_count,
        "validation_status": (
            "PASSED"
            if invalid_source_company_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "VALID_RECORD_ORIGIN",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": invalid_origin_count,
        "validation_status": (
            "PASSED"
            if invalid_origin_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "NON_NEGATIVE_CREDIT_LIMIT",
        "severity": "CRITICAL",
        "rows_checked": input_customer_count,
        "rows_failed": negative_credit_limit_count,
        "validation_status": (
            "PASSED"
            if negative_credit_limit_count == 0
            else "FAILED"
        ),
    },
    {
        "validation_rule": "COUNTRY_CODE_FORMAT",
        "severity": "WARNING",
        "rows_checked": input_customer_count,
        "rows_failed": invalid_country_format_count,
        "validation_status": (
            "PASSED"
            if invalid_country_format_count == 0
            else "WARNING"
        ),
    },
    {
        "validation_rule": "CURRENCY_CODE_FORMAT",
        "severity": "WARNING",
        "rows_checked": input_customer_count,
        "rows_failed": invalid_currency_format_count,
        "validation_status": (
            "PASSED"
            if invalid_currency_format_count == 0
            else "WARNING"
        ),
    },
    {
        "validation_rule": "EMAIL_FORMAT",
        "severity": "WARNING",
        "rows_checked": input_customer_count,
        "rows_failed": invalid_email_format_count,
        "validation_status": (
            "PASSED"
            if invalid_email_format_count == 0
            else "WARNING"
        ),
    },
    {
        "validation_rule": "COUNTRY_CODE_COMPLETENESS",
        "severity": "WARNING",
        "rows_checked": input_customer_count,
        "rows_failed": missing_country_count,
        "validation_status": (
            "PASSED"
            if missing_country_count == 0
            else "WARNING"
        ),
    },
    {
        "validation_rule": "CURRENCY_CODE_COMPLETENESS",
        "severity": "WARNING",
        "rows_checked": input_customer_count,
        "rows_failed": missing_currency_count,
        "validation_status": (
            "PASSED"
            if missing_currency_count == 0
            else "WARNING"
        ),
    },
]


DIM_CUSTOMER_VALIDATION_SUMMARY_DF = (
    spark.createDataFrame(
        validation_summary_rows,
        schema=validation_summary_schema,
    )
)


display(
    DIM_CUSTOMER_VALIDATION_SUMMARY_DF.orderBy(
        "severity",
        "validation_rule",
    )
)


# -----------------------------------------------------------------------------
# 8. Produce the validated customer DataFrame
# -----------------------------------------------------------------------------

CUSTOMER_VALIDATED_DF = (
    CUSTOMER_WITH_COMPANY_DF

    .withColumn(
        "customer_id",
        F.trim(
            F.col("customer_id")
        )
    )

    .withColumn(
        "customer_name",
        F.trim(
            F.col("customer_name")
        )
    )

    .withColumn(
        "country_code",
        F.when(
            F.trim(
                F.col("country_code")
            ) == "",
            F.lit(None),
        ).otherwise(
            F.upper(
                F.trim(
                    F.col("country_code")
                )
            )
        )
    )

    .withColumn(
        "currency_code",
        F.when(
            F.trim(
                F.col("currency_code")
            ) == "",
            F.lit(None),
        ).otherwise(
            F.upper(
                F.trim(
                    F.col("currency_code")
                )
            )
        )
    )

    .withColumn(
        "validation_utc",
        F.lit(
            VALIDATION_UTC
        ).cast(
            "timestamp"
        )
    )
)


# -----------------------------------------------------------------------------
# 9. Print the validation summary
# -----------------------------------------------------------------------------

critical_failure_count = (
    DIM_CUSTOMER_VALIDATION_SUMMARY_DF
    .filter(
        F.col("severity") == "CRITICAL"
    )
    .filter(
        F.col("validation_status") == "FAILED"
    )
    .count()
)


warning_rule_count = (
    DIM_CUSTOMER_VALIDATION_SUMMARY_DF
    .filter(
        F.col("severity") == "WARNING"
    )
    .filter(
        F.col("validation_status") == "WARNING"
    )
    .count()
)


print("=" * 80)
print("DIM_CUSTOMER PRE-WRITE VALIDATION")
print("=" * 80)
print(f"Customer rows checked       : {input_customer_count}")
print(f"Missing business keys       : {missing_business_key_count}")
print(f"Missing customer IDs        : {missing_customer_id_count}")
print(f"Missing customer names      : {missing_customer_name_count}")
print(f"Unknown company mappings    : {unknown_company_count}")
print(f"Duplicate business keys     : {duplicate_business_key_count}")
print(f"Invalid source systems      : {invalid_source_system_count}")
print(f"Invalid source companies    : {invalid_source_company_count}")
print(f"Invalid record origins      : {invalid_origin_count}")
print(f"Negative credit limits      : {negative_credit_limit_count}")
print(f"Country format warnings     : {invalid_country_format_count}")
print(f"Currency format warnings    : {invalid_currency_format_count}")
print(f"Email format warnings       : {invalid_email_format_count}")
print(f"Missing country warnings    : {missing_country_count}")
print(f"Missing currency warnings   : {missing_currency_count}")
print(f"Critical rules failed       : {critical_failure_count}")
print(f"Warning rules raised        : {warning_rule_count}")
print("=" * 80)


# -----------------------------------------------------------------------------
# 10. Display critical failure details
# -----------------------------------------------------------------------------

if unknown_company_count > 0:
    print("Customers without a valid company mapping:")

    display(
        unknown_company_df.select(
            "customer_business_key",
            "customer_id",
            "customer_name",
            "source_system",
            "source_company",
        )
    )


if missing_customer_id_count > 0:
    display(
        missing_customer_id_df
    )


if missing_customer_name_count > 0:
    display(
        missing_customer_name_df
    )


if duplicate_business_key_count > 0:
    display(
        duplicate_business_key_df
    )


# -----------------------------------------------------------------------------
# 11. Stop only for critical failures
# -----------------------------------------------------------------------------

if critical_failure_count > 0:
    raise RuntimeError(
        "DIM_CUSTOMER pre-write validation failed. "
        f"{critical_failure_count} critical validation rule(s) failed."
    )


validated_customer_count = (
    CUSTOMER_VALIDATED_DF.count()
)


if validated_customer_count != input_customer_count:
    raise RuntimeError(
        "Validated customer row count does not match the input row count."
    )


print(
    "DIM_CUSTOMER PRE-WRITE VALIDATION: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fefb4781-f8a7-47a9-8c33-e40cc1f5884c)

DIM_CUSTOMER PRE-WRITE VALIDATION
Customer rows checked       : 981
Missing business keys       : 0
Missing customer IDs        : 0
Missing customer names      : 0
Unknown company mappings    : 0
Duplicate business keys     : 0
Invalid source systems      : 0
Invalid source companies    : 0
Invalid record origins      : 0
Negative credit limits      : 0
Country format warnings     : 0
Currency format warnings    : 0
Email format warnings       : 0
Missing country warnings    : 40
Missing currency warnings   : 0
Critical rules failed       : 0
Warning rules raised        : 1
DIM_CUSTOMER PRE-WRITE VALIDATION: SUCCEEDED


In [9]:
# =============================================================================
# CELL 8 — PREPARE DIM_CUSTOMER SILVER RECORDS
#
# Purpose:
# - Standardise final customer attributes.
# - Create an SCD Type 2 attribute hash.
# - Create version-specific surrogate keys.
# - Add effective dates and Silver audit metadata.
# - Produce DIM_CUSTOMER_PREPARED_DF for the target load.
#
# Important:
# - current_balance is excluded from the SCD Type 2 attribute hash because
#   it is an operational measure that may change frequently.
# =============================================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


# -----------------------------------------------------------------------------
# 1. Validate required upstream objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "CUSTOMER_VALIDATED_DF",
    "PIPELINE_RUN_ID",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 8 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 through 7 first."
    )


# -----------------------------------------------------------------------------
# 2. Use one timestamp for the complete dim_customer load
# -----------------------------------------------------------------------------

DIM_CUSTOMER_LOAD_UTC = datetime.now(
    timezone.utc
).replace(tzinfo=None)


# -----------------------------------------------------------------------------
# 3. Reusable optional-string cleansing helper
# -----------------------------------------------------------------------------

def clean_optional_string(column_name):
    return (
        F.when(
            F.col(column_name).isNull()
            |
            (
                F.length(
                    F.trim(
                        F.col(column_name).cast("string")
                    )
                ) == 0
            ),
            F.lit(None).cast("string"),
        )
        .otherwise(
            F.trim(
                F.col(column_name).cast("string")
            )
        )
    )


# -----------------------------------------------------------------------------
# 4. Standardise final customer attributes
# -----------------------------------------------------------------------------

DIM_CUSTOMER_PREPARED_BASE_DF = (
    CUSTOMER_VALIDATED_DF

    .withColumn(
        "customer_business_key",
        F.upper(
            F.trim(
                F.col("customer_business_key")
            )
        ),
    )

    .withColumn(
        "customer_id",
        F.upper(
            F.trim(
                F.col("customer_id")
            )
        ),
    )

    .withColumn(
        "customer_name",
        F.trim(
            F.col("customer_name")
        ),
    )

    .withColumn(
        "customer_type",
        clean_optional_string(
            "customer_type"
        ),
    )

    .withColumn(
        "country_code",
        F.when(
            clean_optional_string(
                "country_code"
            ).isNull(),
            F.lit(None).cast("string"),
        ).otherwise(
            F.upper(
                clean_optional_string(
                    "country_code"
                )
            )
        ),
    )

    .withColumn(
        "city",
        clean_optional_string(
            "city"
        ),
    )

    .withColumn(
        "postal_code",
        clean_optional_string(
            "postal_code"
        ),
    )

    .withColumn(
        "currency_code",
        F.when(
            clean_optional_string(
                "currency_code"
            ).isNull(),
            F.lit(None).cast("string"),
        ).otherwise(
            F.upper(
                clean_optional_string(
                    "currency_code"
                )
            )
        ),
    )

    .withColumn(
        "payment_terms",
        F.when(
            clean_optional_string(
                "payment_terms"
            ).isNull(),
            F.lit(None).cast("string"),
        ).otherwise(
            F.upper(
                clean_optional_string(
                    "payment_terms"
                )
            )
        ),
    )

    .withColumn(
        "credit_limit",
        F.col("credit_limit").cast(
            DecimalType(18, 2)
        ),
    )

    .withColumn(
        "current_balance",
        F.col("current_balance").cast(
            DecimalType(18, 2)
        ),
    )

    .withColumn(
        "is_active",
        F.coalesce(
            F.col("is_active").cast("boolean"),
            F.lit(True),
        ),
    )

    .withColumn(
        "primary_phone",
        clean_optional_string(
            "primary_phone"
        ),
    )

    .withColumn(
        "email_address",
        F.when(
            clean_optional_string(
                "email_address"
            ).isNull(),
            F.lit(None).cast("string"),
        ).otherwise(
            F.lower(
                clean_optional_string(
                    "email_address"
                )
            )
        ),
    )

    .withColumn(
        "parent_customer_id",
        clean_optional_string(
            "parent_customer_id"
        ),
    )

    .withColumn(
        "is_job",
        F.coalesce(
            F.col("is_job").cast("boolean"),
            F.lit(False),
        ),
    )

    .withColumn(
        "source_priority",
        F.col("source_priority").cast("int"),
    )

    .withColumn(
        "source_created_utc",
        F.col("source_created_utc").cast("timestamp"),
    )

    .withColumn(
        "source_modified_utc",
        F.col("source_modified_utc").cast("timestamp"),
    )
)


# -----------------------------------------------------------------------------
# 5. Create the SCD Type 2 attribute hash
#
# Included:
# - Descriptive and classification attributes
# - Company relationship
# - Geography and currency
# - Payment and credit attributes
# - Contact information
# - Parent/job classification and active status
#
# Excluded:
# - current_balance
# - Source timestamps
# - Source priority and record origin
# - Silver audit metadata
# -----------------------------------------------------------------------------

DIM_CUSTOMER_HASHED_DF = (
    DIM_CUSTOMER_PREPARED_BASE_DF
    .withColumn(
        "attribute_hash",
        F.sha2(
            F.concat_ws(
                "||",

                F.coalesce(
                    F.col("customer_name"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("customer_type"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("company_key"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("country_code"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("city"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("postal_code"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("currency_code"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("payment_terms"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("credit_limit").cast("string"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("is_active").cast("string"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("primary_phone"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("email_address"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("parent_customer_id"),
                    F.lit(""),
                ),

                F.coalesce(
                    F.col("is_job").cast("string"),
                    F.lit(""),
                ),
            ),
            256,
        ),
    )
)


# -----------------------------------------------------------------------------
# 6. Add SCD Type 2 dates and status
# -----------------------------------------------------------------------------

DIM_CUSTOMER_SCD_DF = (
    DIM_CUSTOMER_HASHED_DF

    .withColumn(
        "effective_from_utc",
        F.lit(
            DIM_CUSTOMER_LOAD_UTC
        ).cast("timestamp"),
    )

    .withColumn(
        "effective_to_utc",
        F.lit(None).cast("timestamp"),
    )

    .withColumn(
        "is_current",
        F.lit(True).cast("boolean"),
    )
)


# -----------------------------------------------------------------------------
# 7. Generate version-specific surrogate keys
#
# customer_business_key stays stable.
# customer_key changes for every new SCD Type 2 version.
# -----------------------------------------------------------------------------

DIM_CUSTOMER_KEYED_DF = (
    DIM_CUSTOMER_SCD_DF
    .withColumn(
        "customer_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("customer_business_key"),
                F.date_format(
                    F.col("effective_from_utc"),
                    "yyyy-MM-dd HH:mm:ss.SSSSSS",
                ),
            ),
            256,
        ),
    )
)


# -----------------------------------------------------------------------------
# 8. Add Silver audit metadata
# -----------------------------------------------------------------------------

DIM_CUSTOMER_AUDITED_DF = (
    DIM_CUSTOMER_KEYED_DF

    .withColumn(
        "silver_created_utc",
        F.lit(
            DIM_CUSTOMER_LOAD_UTC
        ).cast("timestamp"),
    )

    .withColumn(
        "silver_updated_utc",
        F.lit(
            DIM_CUSTOMER_LOAD_UTC
        ).cast("timestamp"),
    )

    .withColumn(
        "pipeline_run_id",
        F.lit(
            PIPELINE_RUN_ID
        ).cast("string"),
    )
)


# -----------------------------------------------------------------------------
# 9. Select the final dim_customer column order
# -----------------------------------------------------------------------------

DIM_CUSTOMER_PREPARED_DF = (
    DIM_CUSTOMER_AUDITED_DF
    .select(
        "customer_key",
        "customer_business_key",

        "customer_id",
        "customer_name",
        "customer_type",

        "company_key",
        "company_business_key",
        "company_code",

        "country_code",
        "city",
        "postal_code",

        "currency_code",
        "payment_terms",
        "credit_limit",

        # Retained for reporting, but excluded from the SCD attribute hash.
        "current_balance",

        "is_active",
        "primary_phone",
        "email_address",
        "parent_customer_id",
        "is_job",

        "source_system",
        "source_company",
        "record_origin",
        "source_priority",

        "source_created_utc",
        "source_modified_utc",

        "attribute_hash",

        "effective_from_utc",
        "effective_to_utc",
        "is_current",

        "silver_created_utc",
        "silver_updated_utc",
        "pipeline_run_id",
    )
)


# -----------------------------------------------------------------------------
# 10. Validate prepared records dynamically
# -----------------------------------------------------------------------------

expected_customer_count = (
    CUSTOMER_VALIDATED_DF.count()
)


prepared_customer_count = (
    DIM_CUSTOMER_PREPARED_DF.count()
)


null_customer_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .filter(
        F.col("customer_key").isNull()
        |
        F.col("customer_business_key").isNull()
    )
    .count()
)


null_company_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .filter(
        F.col("company_key").isNull()
    )
    .count()
)


null_attribute_hash_count = (
    DIM_CUSTOMER_PREPARED_DF
    .filter(
        F.col("attribute_hash").isNull()
    )
    .count()
)


duplicate_customer_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .groupBy(
        "customer_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


duplicate_business_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


print("=" * 80)
print("DIM_CUSTOMER RECORD PREPARATION")
print("=" * 80)
print(
    f"Expected records        : "
    f"{expected_customer_count}"
)
print(
    f"Records prepared        : "
    f"{prepared_customer_count}"
)
print(
    f"Null customer keys      : "
    f"{null_customer_key_count}"
)
print(
    f"Null company keys       : "
    f"{null_company_key_count}"
)
print(
    f"Duplicate customer keys : "
    f"{duplicate_customer_key_count}"
)
print(
    f"Duplicate business keys : "
    f"{duplicate_business_key_count}"
)
print(
    f"Null attribute hashes   : "
    f"{null_attribute_hash_count}"
)
print(
    f"Load timestamp          : "
    f"{DIM_CUSTOMER_LOAD_UTC}"
)
print("=" * 80)


# -----------------------------------------------------------------------------
# 11. Enforce preparation integrity
# -----------------------------------------------------------------------------

if prepared_customer_count != expected_customer_count:
    raise RuntimeError(
        "The prepared customer count does not match "
        "the validated source count."
    )


if null_customer_key_count > 0:
    raise RuntimeError(
        f"{null_customer_key_count} prepared customer records "
        "have null customer keys."
    )


if null_company_key_count > 0:
    raise RuntimeError(
        f"{null_company_key_count} prepared customer records "
        "have no company key."
    )


if duplicate_customer_key_count > 0:
    raise RuntimeError(
        f"{duplicate_customer_key_count} duplicate customer "
        "surrogate keys were generated."
    )


if duplicate_business_key_count > 0:
    raise RuntimeError(
        f"{duplicate_business_key_count} duplicate customer "
        "business keys exist."
    )


if null_attribute_hash_count > 0:
    raise RuntimeError(
        f"{null_attribute_hash_count} prepared customer records "
        "have null attribute hashes."
    )


# -----------------------------------------------------------------------------
# 12. Display representative prepared records
# -----------------------------------------------------------------------------

display(
    DIM_CUSTOMER_PREPARED_DF
    .orderBy(
        "source_system",
        "record_origin",
        "customer_id",
    )
    .limit(100)
)


print(
    "DIM_CUSTOMER RECORD PREPARATION: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 11, Finished, Available, Finished, False)

DIM_CUSTOMER RECORD PREPARATION
Expected records        : 981
Records prepared        : 981
Null customer keys      : 0
Null company keys       : 0
Duplicate customer keys : 0
Duplicate business keys : 0
Null attribute hashes   : 0
Load timestamp          : 2026-08-12 14:24:34.580935


SynapseWidget(Synapse.DataFrame, 4cc841bc-bab5-4614-b5f7-3c8d4a206677)

DIM_CUSTOMER RECORD PREPARATION: SUCCEEDED


In [10]:
# =============================================================================
# CELL 9 — LOAD DIM_CUSTOMER USING SCD TYPE 2
#
# Initial run:
# - Creates dbo.dim_customer.
#
# Subsequent runs:
# - Leaves unchanged customers untouched.
# - Expires changed current versions.
# - Inserts new versions of changed customers.
# - Inserts completely new customers.
#
# Important:
# - Run this notebook as the only writer to dbo.dim_customer.
# - This cell uses dynamic source counts and does not assume a fixed total.
# =============================================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# 1. Validate required upstream objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "DIM_CUSTOMER_PREPARED_DF",
    "DIM_CUSTOMER_LOAD_UTC",
    "PIPELINE_RUN_ID",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 9 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 through 8 first."
    )


TARGET_TABLE = "dbo.dim_customer"


# -----------------------------------------------------------------------------
# 2. Validate the prepared source
# -----------------------------------------------------------------------------

source_record_count = (
    DIM_CUSTOMER_PREPARED_DF.count()
)


if source_record_count == 0:
    raise RuntimeError(
        "DIM_CUSTOMER_PREPARED_DF contains no records. "
        "The dim_customer load cannot continue."
    )


duplicate_source_business_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


duplicate_source_surrogate_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .groupBy(
        "customer_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


null_source_key_count = (
    DIM_CUSTOMER_PREPARED_DF
    .filter(
        F.col("customer_key").isNull()
        |
        F.col("customer_business_key").isNull()
    )
    .count()
)


null_source_hash_count = (
    DIM_CUSTOMER_PREPARED_DF
    .filter(
        F.col("attribute_hash").isNull()
    )
    .count()
)


if duplicate_source_business_key_count > 0:
    raise RuntimeError(
        f"{duplicate_source_business_key_count} duplicate customer "
        "business keys exist in the prepared source."
    )


if duplicate_source_surrogate_key_count > 0:
    raise RuntimeError(
        f"{duplicate_source_surrogate_key_count} duplicate customer "
        "surrogate keys exist in the prepared source."
    )


if null_source_key_count > 0:
    raise RuntimeError(
        f"{null_source_key_count} prepared customer records "
        "contain null keys."
    )


if null_source_hash_count > 0:
    raise RuntimeError(
        f"{null_source_hash_count} prepared customer records "
        "contain null attribute hashes."
    )


# Materialise because the prepared DataFrame is referenced several times.
DIM_CUSTOMER_LOAD_SOURCE_DF = (
    DIM_CUSTOMER_PREPARED_DF.cache()
)

DIM_CUSTOMER_LOAD_SOURCE_DF.count()


# -----------------------------------------------------------------------------
# 3. Determine whether the target already exists
# -----------------------------------------------------------------------------

target_exists = spark.catalog.tableExists(
    TARGET_TABLE
)


if not target_exists:

    # -------------------------------------------------------------------------
    # Initial dimension creation
    # -------------------------------------------------------------------------

    (
        DIM_CUSTOMER_LOAD_SOURCE_DF
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true",
        )
        .saveAsTable(
            TARGET_TABLE
        )
    )


    new_customer_count = source_record_count
    changed_customer_count = 0
    unchanged_customer_count = 0
    expired_customer_count = 0
    inserted_customer_count = source_record_count

    load_action = "INITIAL_CREATE"


else:

    # -------------------------------------------------------------------------
    # Read current target versions for comparison
    # -------------------------------------------------------------------------

    current_target_df = (
        spark.table(
            TARGET_TABLE
        )
        .filter(
            F.col("is_current") == F.lit(True)
        )
        .select(
            "customer_business_key",
            "attribute_hash",
        )
        .cache()
    )


    current_target_count = (
        current_target_df.count()
    )


    # -------------------------------------------------------------------------
    # Validate target integrity before changing it
    # -------------------------------------------------------------------------

    duplicate_target_current_count = (
        current_target_df
        .groupBy(
            "customer_business_key"
        )
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )


    null_target_business_key_count = (
        current_target_df
        .filter(
            F.col("customer_business_key").isNull()
        )
        .count()
    )


    null_target_attribute_hash_count = (
        current_target_df
        .filter(
            F.col("attribute_hash").isNull()
        )
        .count()
    )


    if duplicate_target_current_count > 0:
        raise RuntimeError(
            "The existing dim_customer table contains more than "
            "one current row for a customer business key."
        )


    if null_target_business_key_count > 0:
        raise RuntimeError(
            f"The existing dim_customer table contains "
            f"{null_target_business_key_count} current rows "
            "with null customer business keys."
        )


    if null_target_attribute_hash_count > 0:
        raise RuntimeError(
            f"The existing dim_customer table contains "
            f"{null_target_attribute_hash_count} current rows "
            "with null attribute hashes."
        )


    # -------------------------------------------------------------------------
    # Compare incoming source with current target records
    # -------------------------------------------------------------------------

    customer_comparison_df = (
        DIM_CUSTOMER_LOAD_SOURCE_DF.alias(
            "source"
        )
        .join(
            current_target_df.alias(
                "target"
            ),
            on="customer_business_key",
            how="left",
        )
        .select(
            "source.*",
            F.col(
                "target.attribute_hash"
            ).alias(
                "target_attribute_hash"
            ),
        )
        .cache()
    )


    # -------------------------------------------------------------------------
    # Classify incoming records
    # -------------------------------------------------------------------------

    NEW_CUSTOMER_DF = (
        customer_comparison_df
        .filter(
            F.col(
                "target_attribute_hash"
            ).isNull()
        )
        .drop(
            "target_attribute_hash"
        )
        .cache()
    )


    CHANGED_CUSTOMER_DF = (
        customer_comparison_df
        .filter(
            F.col(
                "target_attribute_hash"
            ).isNotNull()
            &
            (
                F.col("attribute_hash")
                !=
                F.col("target_attribute_hash")
            )
        )
        .drop(
            "target_attribute_hash"
        )
        .cache()
    )


    UNCHANGED_CUSTOMER_DF = (
        customer_comparison_df
        .filter(
            F.col(
                "target_attribute_hash"
            ).isNotNull()
            &
            (
                F.col("attribute_hash")
                ==
                F.col("target_attribute_hash")
            )
        )
        .drop(
            "target_attribute_hash"
        )
        .cache()
    )


    new_customer_count = (
        NEW_CUSTOMER_DF.count()
    )

    changed_customer_count = (
        CHANGED_CUSTOMER_DF.count()
    )

    unchanged_customer_count = (
        UNCHANGED_CUSTOMER_DF.count()
    )


    # -------------------------------------------------------------------------
    # Reconcile classified source rows
    # -------------------------------------------------------------------------

    classified_record_count = (
        new_customer_count
        + changed_customer_count
        + unchanged_customer_count
    )


    if classified_record_count != source_record_count:
        raise RuntimeError(
            "Customer change-classification reconciliation failed. "
            f"Source={source_record_count}; "
            f"classified={classified_record_count}."
        )


    target_delta = DeltaTable.forName(
        spark,
        TARGET_TABLE,
    )


    # -------------------------------------------------------------------------
    # Expire current versions of changed customers
    # -------------------------------------------------------------------------

    if changed_customer_count > 0:

        changed_customer_keys_df = (
            CHANGED_CUSTOMER_DF
            .select(
                "customer_business_key"
            )
            .distinct()
        )


        (
            target_delta.alias(
                "target"
            )
            .merge(
                changed_customer_keys_df.alias(
                    "source"
                ),
                """
                target.customer_business_key =
                    source.customer_business_key
                AND target.is_current = true
                """,
            )
            .whenMatchedUpdate(
                set={
                    "effective_to_utc":
                        f"timestamp'{DIM_CUSTOMER_LOAD_UTC}'",

                    "is_current":
                        "false",

                    "silver_updated_utc":
                        f"timestamp'{DIM_CUSTOMER_LOAD_UTC}'",

                    "pipeline_run_id":
                        f"'{PIPELINE_RUN_ID}'",
                }
            )
            .execute()
        )


    # -------------------------------------------------------------------------
    # Insert new customers and new versions of changed customers
    # -------------------------------------------------------------------------

    CUSTOMERS_TO_INSERT_DF = (
        NEW_CUSTOMER_DF
        .unionByName(
            CHANGED_CUSTOMER_DF
        )
    )


    inserted_customer_count = (
        CUSTOMERS_TO_INSERT_DF.count()
    )


    if inserted_customer_count > 0:

        (
            CUSTOMERS_TO_INSERT_DF
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(
                TARGET_TABLE
            )
        )


    expired_customer_count = (
        changed_customer_count
    )


    if changed_customer_count > 0:
        load_action = "SCD2_CHANGES_APPLIED"

    elif new_customer_count > 0:
        load_action = "NEW_CUSTOMERS_INSERTED"

    else:
        load_action = "NO_CHANGES"


    # -------------------------------------------------------------------------
    # Release cached intermediate DataFrames
    # -------------------------------------------------------------------------

    current_target_df.unpersist()
    customer_comparison_df.unpersist()
    NEW_CUSTOMER_DF.unpersist()
    CHANGED_CUSTOMER_DF.unpersist()
    UNCHANGED_CUSTOMER_DF.unpersist()


# -----------------------------------------------------------------------------
# 4. Validate the result of this load
# -----------------------------------------------------------------------------

DIM_CUSTOMER_TARGET_AFTER_LOAD_DF = (
    spark.table(
        TARGET_TABLE
    )
)


current_target_after_load_count = (
    DIM_CUSTOMER_TARGET_AFTER_LOAD_DF
    .filter(
        F.col("is_current") == F.lit(True)
    )
    .count()
)


duplicate_current_after_load_count = (
    DIM_CUSTOMER_TARGET_AFTER_LOAD_DF
    .filter(
        F.col("is_current") == F.lit(True)
    )
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if current_target_after_load_count < source_record_count:
    raise RuntimeError(
        "The number of current dim_customer records after loading "
        "is lower than the incoming source-record count."
    )


if duplicate_current_after_load_count > 0:
    raise RuntimeError(
        "Duplicate current customer business keys exist after loading."
    )


# -----------------------------------------------------------------------------
# 5. Display load summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("DIM_CUSTOMER SCD TYPE 2 LOAD")
print("=" * 80)
print(
    f"Target table        : "
    f"{TARGET_TABLE}"
)
print(
    f"Load action         : "
    f"{load_action}"
)
print(
    f"Source records      : "
    f"{source_record_count}"
)
print(
    f"New customers       : "
    f"{new_customer_count}"
)
print(
    f"Changed customers   : "
    f"{changed_customer_count}"
)
print(
    f"Unchanged customers : "
    f"{unchanged_customer_count}"
)
print(
    f"Records inserted    : "
    f"{inserted_customer_count}"
)
print(
    f"Records expired     : "
    f"{expired_customer_count}"
)
print(
    f"Current target rows : "
    f"{current_target_after_load_count}"
)
print(
    "Load status         : SUCCEEDED"
)
print("=" * 80)


DIM_CUSTOMER_LOAD_SOURCE_DF.unpersist()

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 12, Finished, Available, Finished, False)

DIM_CUSTOMER SCD TYPE 2 LOAD
Target table        : dbo.dim_customer
Load action         : NO_CHANGES
Source records      : 981
New customers       : 0
Changed customers   : 0
Unchanged customers : 981
Records inserted    : 0
Records expired     : 0
Current target rows : 981
Load status         : SUCCEEDED


DataFrame[customer_key: string, customer_business_key: string, customer_id: string, customer_name: string, customer_type: string, company_key: string, company_business_key: string, company_code: string, country_code: string, city: string, postal_code: string, currency_code: string, payment_terms: string, credit_limit: decimal(18,2), current_balance: decimal(18,2), is_active: boolean, primary_phone: string, email_address: string, parent_customer_id: string, is_job: boolean, source_system: string, source_company: string, record_origin: string, source_priority: int, source_created_utc: timestamp, source_modified_utc: timestamp, attribute_hash: string, effective_from_utc: timestamp, effective_to_utc: timestamp, is_current: boolean, silver_created_utc: timestamp, silver_updated_utc: timestamp, pipeline_run_id: string]

In [11]:
# =============================================================================
# CELL 10 — DIM_CUSTOMER POST-LOAD VALIDATION
#
# Purpose:
# - Validate the completed dbo.dim_customer table.
# - Confirm current-row counts reconcile to the prepared source.
# - Validate keys, SCD Type 2 dates and mandatory attributes.
# - Confirm every current customer references a current company.
# =============================================================================

from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# 1. Validate required upstream objects
# -----------------------------------------------------------------------------

required_runtime_objects = [
    "DIM_CUSTOMER_PREPARED_DF",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise NameError(
        "Cell 10 cannot run because these upstream objects are missing: "
        + ", ".join(missing_runtime_objects)
        + ". Run Cells 1 through 9 first."
    )


TARGET_TABLE = "dbo.dim_customer"


if not spark.catalog.tableExists(TARGET_TABLE):
    raise RuntimeError(
        f"Required target table does not exist: {TARGET_TABLE}"
    )


# -----------------------------------------------------------------------------
# 2. Read the completed dimension
# -----------------------------------------------------------------------------

DIM_CUSTOMER_DF = spark.table(
    TARGET_TABLE
)


CURRENT_DIM_CUSTOMER_DF = (
    DIM_CUSTOMER_DF
    .filter(
        F.col("is_current") == F.lit(True)
    )
)


HISTORICAL_DIM_CUSTOMER_DF = (
    DIM_CUSTOMER_DF
    .filter(
        F.col("is_current") == F.lit(False)
    )
)


# -----------------------------------------------------------------------------
# 3. Dynamic expected current-customer count
# -----------------------------------------------------------------------------

expected_current_customer_count = (
    DIM_CUSTOMER_PREPARED_DF
    .select(
        "customer_business_key"
    )
    .distinct()
    .count()
)


if expected_current_customer_count == 0:
    raise RuntimeError(
        "The prepared customer source contains no business keys."
    )


# -----------------------------------------------------------------------------
# 4. Core row counts
# -----------------------------------------------------------------------------

total_historical_rows = (
    DIM_CUSTOMER_DF.count()
)


current_customer_count = (
    CURRENT_DIM_CUSTOMER_DF.count()
)


historical_customer_count = (
    HISTORICAL_DIM_CUSTOMER_DF.count()
)


# -----------------------------------------------------------------------------
# 5. Key validation
# -----------------------------------------------------------------------------

null_customer_key_df = (
    DIM_CUSTOMER_DF
    .filter(
        F.col("customer_key").isNull()
        |
        F.col("customer_business_key").isNull()
    )
)


null_customer_key_count = (
    null_customer_key_df.count()
)


null_company_key_df = (
    DIM_CUSTOMER_DF
    .filter(
        F.col("company_key").isNull()
    )
)


null_company_key_count = (
    null_company_key_df.count()
)


null_attribute_hash_df = (
    DIM_CUSTOMER_DF
    .filter(
        F.col("attribute_hash").isNull()
    )
)


null_attribute_hash_count = (
    null_attribute_hash_df.count()
)


duplicate_current_key_df = (
    CURRENT_DIM_CUSTOMER_DF
    .groupBy(
        "customer_business_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_current_key_count = (
    duplicate_current_key_df.count()
)


duplicate_surrogate_key_df = (
    DIM_CUSTOMER_DF
    .groupBy(
        "customer_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_surrogate_key_count = (
    duplicate_surrogate_key_df.count()
)


# -----------------------------------------------------------------------------
# 6. Validate exactly one current version per business key
# -----------------------------------------------------------------------------

invalid_current_version_count_df = (
    DIM_CUSTOMER_DF
    .groupBy(
        "customer_business_key"
    )
    .agg(
        F.sum(
            F.when(
                F.col("is_current") == F.lit(True),
                F.lit(1),
            ).otherwise(
                F.lit(0)
            )
        ).alias(
            "current_version_count"
        )
    )
    .filter(
        F.col("current_version_count") != 1
    )
)


invalid_current_version_count = (
    invalid_current_version_count_df.count()
)


# -----------------------------------------------------------------------------
# 7. SCD Type 2 date validation
# -----------------------------------------------------------------------------

invalid_current_date_df = (
    CURRENT_DIM_CUSTOMER_DF
    .filter(
        F.col("effective_from_utc").isNull()
        |
        F.col("effective_to_utc").isNotNull()
    )
)


invalid_current_date_count = (
    invalid_current_date_df.count()
)


invalid_historical_date_df = (
    HISTORICAL_DIM_CUSTOMER_DF
    .filter(
        F.col("effective_from_utc").isNull()
        |
        F.col("effective_to_utc").isNull()
        |
        (
            F.col("effective_to_utc")
            <=
            F.col("effective_from_utc")
        )
    )
)


invalid_historical_date_count = (
    invalid_historical_date_df.count()
)


# -----------------------------------------------------------------------------
# 8. Referential integrity against current dim_company
# -----------------------------------------------------------------------------

if not spark.catalog.tableExists(
    "dbo.dim_company"
):
    raise RuntimeError(
        "Required dependency dbo.dim_company does not exist."
    )


CURRENT_COMPANY_KEYS_DF = (
    spark.table(
        "dbo.dim_company"
    )
    .filter(
        F.col("is_current") == F.lit(True)
    )
    .select(
        "company_key"
    )
    .distinct()
)


orphan_customer_company_df = (
    CURRENT_DIM_CUSTOMER_DF
    .select(
        "customer_business_key",
        "customer_id",
        "customer_name",
        "company_key",
        "source_system",
        "source_company",
    )
    .join(
        CURRENT_COMPANY_KEYS_DF,
        on="company_key",
        how="left_anti",
    )
)


orphan_customer_company_count = (
    orphan_customer_company_df.count()
)


# -----------------------------------------------------------------------------
# 9. Mandatory current-attribute validation
# -----------------------------------------------------------------------------

invalid_current_customer_df = (
    CURRENT_DIM_CUSTOMER_DF
    .filter(
        F.col("customer_id").isNull()
        |
        (
            F.length(
                F.trim(
                    F.col("customer_id")
                )
            ) == 0
        )
        |
        F.col("customer_name").isNull()
        |
        (
            F.length(
                F.trim(
                    F.col("customer_name")
                )
            ) == 0
        )
        |
        F.col("source_system").isNull()
        |
        F.col("source_company").isNull()
        |
        F.col("record_origin").isNull()
        |
        F.col("company_key").isNull()
    )
)


invalid_current_customer_count = (
    invalid_current_customer_df.count()
)


# -----------------------------------------------------------------------------
# 10. Format and monetary validation
# -----------------------------------------------------------------------------

invalid_country_format_df = (
    CURRENT_DIM_CUSTOMER_DF
    .filter(
        F.col("country_code").isNotNull()
        &
        (
            ~F.upper(
                F.trim(
                    F.col("country_code")
                )
            ).rlike("^[A-Z]{2}$")
        )
    )
)


invalid_country_format_count = (
    invalid_country_format_df.count()
)


invalid_currency_format_df = (
    CURRENT_DIM_CUSTOMER_DF
    .filter(
        F.col("currency_code").isNotNull()
        &
        (
            ~F.upper(
                F.trim(
                    F.col("currency_code")
                )
            ).rlike("^[A-Z]{3}$")
        )
    )
)


invalid_currency_format_count = (
    invalid_currency_format_df.count()
)


negative_credit_limit_df = (
    CURRENT_DIM_CUSTOMER_DF
    .filter(
        F.col("credit_limit") < F.lit(0)
    )
)


negative_credit_limit_count = (
    negative_credit_limit_df.count()
)


negative_current_balance_df = (
    CURRENT_DIM_CUSTOMER_DF
    .filter(
        F.col("current_balance") < F.lit(0)
    )
)


negative_current_balance_count = (
    negative_current_balance_df.count()
)


# -----------------------------------------------------------------------------
# 11. Print validation summary
# -----------------------------------------------------------------------------

print("=" * 80)
print("DIM_CUSTOMER POST-LOAD VALIDATION")
print("=" * 80)
print(
    f"Expected current rows       : "
    f"{expected_current_customer_count}"
)
print(
    f"Total historical rows       : "
    f"{total_historical_rows}"
)
print(
    f"Current customer rows       : "
    f"{current_customer_count}"
)
print(
    f"Historical customer rows    : "
    f"{historical_customer_count}"
)
print(
    f"Duplicate current keys      : "
    f"{duplicate_current_key_count}"
)
print(
    f"Duplicate surrogate keys    : "
    f"{duplicate_surrogate_key_count}"
)
print(
    f"Null customer keys          : "
    f"{null_customer_key_count}"
)
print(
    f"Null company keys           : "
    f"{null_company_key_count}"
)
print(
    f"Null attribute hashes       : "
    f"{null_attribute_hash_count}"
)
print(
    f"Invalid current versions    : "
    f"{invalid_current_version_count}"
)
print(
    f"Invalid current SCD dates   : "
    f"{invalid_current_date_count}"
)
print(
    f"Invalid historical dates    : "
    f"{invalid_historical_date_count}"
)
print(
    f"Orphan company references   : "
    f"{orphan_customer_company_count}"
)
print(
    f"Invalid current customers   : "
    f"{invalid_current_customer_count}"
)
print(
    f"Invalid country formats     : "
    f"{invalid_country_format_count}"
)
print(
    f"Invalid currency formats    : "
    f"{invalid_currency_format_count}"
)
print(
    f"Negative credit limits      : "
    f"{negative_credit_limit_count}"
)
print(
    f"Negative current balances   : "
    f"{negative_current_balance_count}"
)
print("=" * 80)


# -----------------------------------------------------------------------------
# 12. Display failure details
# -----------------------------------------------------------------------------

if duplicate_current_key_count > 0:
    display(
        duplicate_current_key_df
    )


if duplicate_surrogate_key_count > 0:
    display(
        duplicate_surrogate_key_df
    )


if null_customer_key_count > 0:
    display(
        null_customer_key_df
    )


if null_company_key_count > 0:
    display(
        null_company_key_df
    )


if null_attribute_hash_count > 0:
    display(
        null_attribute_hash_df
    )


if invalid_current_version_count > 0:
    display(
        invalid_current_version_count_df
    )


if invalid_current_date_count > 0:
    display(
        invalid_current_date_df
    )


if invalid_historical_date_count > 0:
    display(
        invalid_historical_date_df
    )


if orphan_customer_company_count > 0:
    display(
        orphan_customer_company_df
    )


if invalid_current_customer_count > 0:
    display(
        invalid_current_customer_df
    )


if invalid_country_format_count > 0:
    display(
        invalid_country_format_df
    )


if invalid_currency_format_count > 0:
    display(
        invalid_currency_format_df
    )


if negative_credit_limit_count > 0:
    display(
        negative_credit_limit_df
    )


if negative_current_balance_count > 0:
    display(
        negative_current_balance_df
    )


# -----------------------------------------------------------------------------
# 13. Enforce critical post-load validation
# -----------------------------------------------------------------------------

validation_failures = []


if (
    current_customer_count
    != expected_current_customer_count
):
    validation_failures.append(
        "CURRENT_CUSTOMER_COUNT"
    )


if duplicate_current_key_count > 0:
    validation_failures.append(
        "DUPLICATE_CURRENT_BUSINESS_KEY"
    )


if duplicate_surrogate_key_count > 0:
    validation_failures.append(
        "DUPLICATE_SURROGATE_KEY"
    )


if null_customer_key_count > 0:
    validation_failures.append(
        "NULL_CUSTOMER_KEY"
    )


if null_company_key_count > 0:
    validation_failures.append(
        "NULL_COMPANY_KEY"
    )


if null_attribute_hash_count > 0:
    validation_failures.append(
        "NULL_ATTRIBUTE_HASH"
    )


if invalid_current_version_count > 0:
    validation_failures.append(
        "INVALID_CURRENT_VERSION_COUNT"
    )


if invalid_current_date_count > 0:
    validation_failures.append(
        "INVALID_CURRENT_SCD_DATE"
    )


if invalid_historical_date_count > 0:
    validation_failures.append(
        "INVALID_HISTORICAL_SCD_DATE"
    )


if orphan_customer_company_count > 0:
    validation_failures.append(
        "ORPHAN_COMPANY_REFERENCE"
    )


if invalid_current_customer_count > 0:
    validation_failures.append(
        "INVALID_CURRENT_CUSTOMER"
    )


if invalid_country_format_count > 0:
    validation_failures.append(
        "INVALID_COUNTRY_FORMAT"
    )


if invalid_currency_format_count > 0:
    validation_failures.append(
        "INVALID_CURRENCY_FORMAT"
    )


if negative_credit_limit_count > 0:
    validation_failures.append(
        "NEGATIVE_CREDIT_LIMIT"
    )


if negative_current_balance_count > 0:
    validation_failures.append(
        "NEGATIVE_CURRENT_BALANCE"
    )


if validation_failures:
    raise RuntimeError(
        "DIM_CUSTOMER post-load validation failed: "
        + ", ".join(validation_failures)
    )


# -----------------------------------------------------------------------------
# 14. Display representative current records
# -----------------------------------------------------------------------------

display(
    CURRENT_DIM_CUSTOMER_DF
    .orderBy(
        "source_system",
        "record_origin",
        "customer_id",
    )
    .limit(100)
)


print(
    "DIM_CUSTOMER POST-LOAD VALIDATION: SUCCEEDED"
)

StatementMeta(, f9199f25-29e3-4fa3-b529-bae412fa5a4f, 13, Finished, Available, Finished, False)

DIM_CUSTOMER POST-LOAD VALIDATION
Expected current rows       : 981
Total historical rows       : 1961
Current customer rows       : 981
Historical customer rows    : 980
Duplicate current keys      : 0
Duplicate surrogate keys    : 0
Null customer keys          : 0
Null company keys           : 0
Null attribute hashes       : 0
Invalid current versions    : 0
Invalid current SCD dates   : 0
Invalid historical dates    : 0
Orphan company references   : 0
Invalid current customers   : 0
Invalid country formats     : 0
Invalid currency formats    : 0
Negative credit limits      : 0
Negative current balances   : 0


SynapseWidget(Synapse.DataFrame, ae66f6d1-6d8a-47ef-a423-12f1a7869768)

DIM_CUSTOMER POST-LOAD VALIDATION: SUCCEEDED
